# Set-up

In [ ]:
import sys

assert sys.version_info >= (3, 7)

In [ ]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "rnn"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

# Intro

* Humans naturally perform **prediction tasks** (e.g., anticipating words, events, patterns)

* **Recurrent Neural Networks (RNNs)**:

  * Specialized neural networks designed for **sequence data**
  * Can **predict future values** based on past patterns
  * Assumption: **future follows similar patterns as past**

* **Applications of RNNs (time series + sequences)**:

  * Daily active users (web analytics)
  * Temperature forecasting
  * Power consumption prediction
  * Vehicle trajectory prediction
  * Natural Language Processing (NLP):

    * Machine translation
    * Speech-to-text
  * Audio and text processing

* Key capability:

  * Handles **variable-length sequences** (unlike traditional models needing fixed-size input)

* Training method:

  * Uses **Backpropagation Through Time (BPTT)**

    * Extension of backpropagation for sequential data

* Time series forecasting:

  * RNNs learn **patterns over time** → use them for prediction

* Baseline models:

  * **ARMA (AutoRegressive Moving Average)** models

    * Traditional statistical models for time series
    * Used for comparison with RNN performance

* Main challenges in RNNs:

  * **Unstable gradients**:

    * Vanishing gradients
    * Exploding gradients
    * Solutions:

      * Recurrent dropout
      * Recurrent layer normalization

  * **Limited short-term memory**:

    * Difficulty remembering long-term dependencies
    * Solutions:

      * **LSTM (Long Short-Term Memory)**
      * **GRU (Gated Recurrent Unit)**

* Alternatives to RNNs:

  * **Dense (Fully Connected) Networks**:

    * Suitable for **small sequences**

  * **Convolutional Neural Networks (CNNs)**:

    * Can handle **very long sequences efficiently**
    * Used in sequence modeling (e.g., audio, text)

* Advanced architecture:

  * **WaveNet**:

    * CNN-based model
    * Handles **very long sequences (thousands of time steps)**
    * Used for sequence generation and prediction

* Key takeaway:

  * RNNs are powerful for **sequential and time-dependent data**, but have **limitations** that require advanced architectures (LSTM, GRU) or alternatives (CNNs)


# Recurrent Neurons and Layers

* **Feedforward Neural Networks (FNNs)**:

  * Information flows **only forward**:

    * Input → Hidden layers → Output
  * No feedback or memory of previous inputs

* **Recurrent Neural Networks (RNNs)**:

  * Similar to feedforward networks but with **feedback connections**
  * Output is **fed back into the network**
  * Enables **memory of previous time steps**

* **Basic RNN (single neuron)**:

  * At time step ( t ):

    * Takes:

      * Current input: ( $x^{(t)}$ )
      * Previous output: ( $\hat{y}^{(t-1)}$ )
    * Produces:

      * Current output: ( $\hat{y}^{(t)}$ )

* **Initial condition**:

  * At ( t = 0 ):

    * No previous output
    * Set:

      * ( $\hat{y}^{(0)}$ = 0 )

* **Key idea**:

  * Output depends on:

    * Current input
    * **Past information (memory)**

* **Unrolling through time**:

  * RNN can be visualized as:

    * Same neuron repeated across time steps
  * Each step:

    * Shares the **same weights**
  * This representation helps in:

    * Understanding computation
    * Training with BPTT

* **RNN layer (multiple neurons)**:

  * Instead of one neuron:

    * Use a **layer of neurons**
  * At each time step:

    * Input becomes a **vector**:

      * ( $\mathbf{x}^{(t)}$ )
    * Output becomes a **vector**:

      * ( $\hat{\mathbf{y}}^{(t)}$ )

* **Connections in RNN layer**:

  * Each neuron receives:

    * Current input vector: ( $\mathbf{x}^{(t)}$ )
    * Previous output vector: ( $\hat{\mathbf{y}}^{(t-1)}$ )

* **Weights in RNN**:

  * Two types of weights:

    * Input weights:

      * ( $\mathbf{W}_x$ ) → connects input to hidden/output
    * Recurrent weights:

      * ( $\mathbf{W}_y$ ) → connects previous output to current step

* **Matrix representation (for full layer)**:

  * Instead of individual weight vectors:

    * Use matrices:

      * ( $\mathbf{W}_x$ ) for inputs
      * ( $\mathbf{W}_y$ ) for recurrent connections

* **Core takeaway**:

  * RNN introduces **time dependency + memory**
  * Same network is reused across time with **shared weights**
  * Enables modeling of **sequential data** efficiently


* **Output of RNN (single instance at time step ( t ))**:

$$\hat{y}^{(t)} = \phi\left(W_x^T x^{(t)} + W_y^T \hat{y}^{(t-1)} + b\right)$$

* Components of the equation:

  * ( $x^{(t)}$ ): input at time step ( t )
  * ( $\hat{y}^{(t-1)}$ ): output from previous time step (memory)
  * ( $W_x$ ): weights for current input
  * ( $W_y$ ): weights for previous output (recurrent weights)
  * ( b ): bias vector
  * ( $\phi(\cdot)$ ): activation function (e.g., ReLU, tanh)

* Key idea:

  * Output depends on:

    * **current input**
    * **previous output (memory)**
  * This creates **temporal dependency**

---

* **Mini-batch computation (vectorized form)**:

$$\hat{Y}^{(t)} = \phi\left(X^{(t)} W_x + \hat{Y}^{(t-1)} W_y + b\right)$$

* Matrix dimensions:

  * $( X^{(t)} ): ( m \times n_{inputs} )$
  * $( \hat{Y}^{(t)} ): ( m \times n_{neurons} )$
  * $( W_x ): ( n_{inputs} \times n_{neurons} )$
  * $( W_y ): ( n_{neurons} \times n_{neurons} )$
  * ( b ): vector of size ( $n_{neurons}$ )
  * ( m ): number of instances (batch size)

---

* **Concatenated form (optimized computation)**:

$$\hat{Y}^{(t)} = \phi\left([X^{(t)} \ \hat{Y}^{(t-1)}] W + b\right)$$

* Where:

  * $( [X^{(t)} \ \hat{Y}^{(t-1)}] )$: horizontal concatenation
  * ( W ): combined weight matrix of shape
    $( (n_{inputs} + n_{neurons}) \times n_{neurons} )$

* Benefit:

  * Simplifies computation into **single matrix multiplication**
  * More efficient in practice

---

* **Recursive dependency (very important)**:

  * ( $\hat{Y}^{(t)}$ ) depends on:

    * ( $\hat{Y}^{(t-1)}$ )
    * which depends on ( $\hat{Y}^{(t-2)}$ )
    * and so on...

* Therefore:

  * Output at time ( t ) depends on:

    * **all past inputs**

$$[
X^{(0)}, X^{(1)}, X^{(2)}, \dots, X^{(t)}
]$$

* This is what gives RNNs their **memory capability**

---

* **Initial step (t = 0)**:

  * No previous output available
  * Initialize:

$$[
\hat{Y}^{(0)} = 0
]$$

* This acts as the **starting memory state**

---

* **Core takeaway**:

  * RNN computation is:

    * **recursive + sequential**
  * Each step carries forward information → builds **temporal understanding**
  * Matrix formulation enables **efficient training with mini-batches**


## Memory Cells

* **RNN as a memory system**:

  * Output at time ( t ) depends on **all previous inputs**
  * This gives RNNs a form of **memory over time**

* **Memory cell (cell)**:

  * Any part of a neural network that **preserves state across time steps**
  * Examples:

    * Single recurrent neuron
    * Layer of recurrent neurons

* **Limitation of basic RNN cells**:

  * Can learn only **short-term dependencies**
  * Typical range:

    * Around **~10 time steps** (task-dependent)
  * Struggle with long-term patterns

* **Advanced cells (mentioned ahead)**:

  * Designed to capture **longer dependencies**
  * Can learn patterns:

    * Roughly **10× longer than basic RNNs**
  * Examples (later in chapter):

    * LSTM
    * GRU

---

* **Hidden state (core concept)**:

  * Denoted as:

$$[
h^{(t)}
]$$

* Represents the **internal memory/state** of the cell at time ( t )

* **State update rule**:

$$h^{(t)} = f\left(x^{(t)}, h^{(t-1)}\right)$$

* Meaning:

  * Current state depends on:

    * Current input ( $x^{(t)}$ )
    * Previous state ( $h^{(t-1)}$ )

---

* **Output of the cell**:

  * Denoted as:

$$[
\hat{y}^{(t)}
]$$

* Computed using:

  * Current input
  * Previous hidden state

* **Basic RNN case**:

  * Output = hidden state:

$$[
\hat{y}^{(t)} = h^{(t)}
]$$

* **Advanced cells (LSTM, GRU)**:

  * Output is **not always equal** to hidden state
  * They maintain **more complex internal structures**

---

* **Key intuition**:

  * Hidden state ( $h^{(t)}$ ) acts like:

    * A **running summary of past information**
  * Each time step:

    * Updates this summary with new input

---

* **Core takeaway**:

  * RNNs rely on **hidden state (memory) propagation**
  * Basic cells → **short memory**
  * Advanced cells → **long-term memory handling**


## Input and Output Sequences

* **RNN versatility**:

  * Can handle **different input-output sequence structures**
  * Not limited to one fixed mapping

---

* **Sequence-to-sequence (many-to-many)**:

  * Input: sequence → Output: sequence
  * Output at each time step
  * Example:

    * Time series forecasting
    * Input: last ( N ) days data
    * Output: values shifted by 1 step (predict future)

* Key idea:

  * Predict **next step for each time step**

---

* **Sequence-to-vector (many-to-one)**:

  * Input: sequence → Output: single value
  * Only **last output is used**
  * Example:

    * Sentiment analysis

      * Input: sentence/review
      * Output: sentiment score (0 → 1)

* Key idea:

  * Entire sequence compressed into **single representation**

---

* **Vector-to-sequence (one-to-many)**:

  * Input: single vector → Output: sequence
  * Same input repeated across time steps
  * Example:

    * Image captioning

      * Input: image (or CNN features)
      * Output: sequence of words (caption)

* Key idea:

  * Generate **sequence from fixed input**

---

* **Encoder–Decoder architecture**:

  * Combines:

    * Sequence-to-vector (encoder)
    * Vector-to-sequence (decoder)

* Workflow:

  * Encoder:

    * Input sequence → compressed into **single vector**
  * Decoder:

    * Takes vector → generates output sequence

* Example:

  * Machine translation

    * Input: sentence (language A)
    * Output: sentence (language B)

---

* **Why encoder–decoder is better than basic seq-to-seq**:

  * Translation depends on **entire sentence context**
  * Early words may depend on **later words**
  * Encoder ensures:

    * Full sentence is understood before decoding

---

* **Important concept**:

  * RNNs can model:

    * Many-to-many
    * Many-to-one
    * One-to-many
    * Encoder–decoder (many-to-many with transformation)

---

* **Core takeaway**:

  * RNNs are highly flexible for **sequence transformations**
  * Choice of architecture depends on:

    * Input type
    * Output requirement
    * Task (forecasting, classification, generation, translation)

---

* **Next logical step**:

  * Understanding **how RNNs are trained**
  * Leads to:

    * **Backpropagation Through Time (BPTT)**


# Training RNNs

* **Training RNNs → Backpropagation Through Time (BPTT)**:

  * Key idea:

    * **Unroll the RNN across time steps**
    * Apply **standard backpropagation** on this expanded network

---

* **Step 1: Forward pass**:

  * Input sequence flows through **unrolled network**
  * At each time step:

    * Compute output ( $\hat{Y}^{(t)}$ )
  * Produces a **sequence of predictions**

---

* **Step 2: Loss computation**:

  * Loss function compares:

    * True outputs: $( Y^{(0)}, Y^{(1)}, ..., Y^{(T)} )$
    * Predictions: $( \hat{Y}^{(0)}, \hat{Y}^{(1)}, ..., \hat{Y}^{(T)} )$

* General form:

$$[
\mathcal{L}(Y^{(0)}, Y^{(1)}, \dots, Y^{(T)};\ \hat{Y}^{(0)}, \hat{Y}^{(1)}, \dots, \hat{Y}^{(T)})
]$$

* Important:

  * Loss may **ignore some time steps**

    * Example:

      * Sequence-to-vector → only last output used
      * Some intermediate outputs may not contribute

---

* **Step 3: Backward pass (BPTT)**:

  * Gradients are propagated:

    * **Backward through time**
    * Across all unrolled steps

* Key behavior:

  * If an output is **not used in loss**:

    * No gradient flows through that step

---

* **Step 4: Shared parameters effect**:

  * Same weights used at every time step:

    * ( W, b ) are **shared**
  * During backprop:

    * Gradients from **all time steps accumulate**
  * Result:

    * Parameters updated using **combined influence of all steps**

---

* **Step 5: Parameter update**:

  * After computing gradients:

    * Apply **gradient descent (or variant)**
  * Same as standard neural networks

---

* **Key intuition**:

  * RNN training =

    * **Feedforward over time**
    * * **Backpropagation over time**

---

* **Important consequences**:

  * Long sequences → deep unrolled network
  * Leads to:

    * **Vanishing gradients**
    * **Exploding gradients**

---

* **Practical note**:

  * Frameworks like **Keras / TensorFlow**:

    * Automatically handle:

      * Unrolling
      * Gradient computation
      * Parameter updates

---

* **Core takeaway**:

  * BPTT allows RNNs to learn **temporal dependencies**
  * Training complexity comes from:

    * **time dimension + shared weights + recursive structure**


# Forecasting a Time Series

* Goal:

  * Build a model to forecast:

    * Next day bus ridership
    * Next day rail ridership
  * Dataset:

    * Daily Chicago transit ridership since 2001

* Data preprocessing steps:

  * Load CSV with Pandas
  * Parse `service_date` as datetime
  * Rename columns:

    * `date`
    * `day_type`
    * `bus`
    * `rail`
    * `total`
  * Sort rows by date
  * Set date as index
  * Remove `total` column:

$$[
\text{total} = \text{bus} + \text{rail}
]$$

* Remove duplicate rows

---

* `day_type` meanings:

  * `W` → Weekday
  * `A` → Saturday
  * `U` → Sunday / Holiday

---

* **Time series**:

  * Data collected across time at regular intervals

* **Multivariate time series**:

  * Multiple values per time step
  * Example:

    * bus + rail together

* **Univariate time series**:

  * Single value per time step
  * Example:

    * only bus ridership

---

* Common time series tasks:

  * Forecasting
  * Imputation (fill missing values)
  * Classification
  * Anomaly detection

---

* **Seasonality**:

  * Repeating patterns over fixed intervals

* In this dataset:

  * Strong **weekly seasonality**
  * Similar ridership patterns repeat every week

---

* **Naive forecasting**:

  * Predict future using past values
  * Here:

$$[
\hat{y}*t = y*{t-7}
]$$

* Forecast tomorrow using value from same weekday last week

* Important:

  * Normally naive forecasting means:

$$[
\hat{y}*t = y*{t-1}
]$$

* But weekly seasonality makes 7-day lag better here

---

* **Lagged time series**:

  * Shifted version of original series
  * Example:

    * 7-day lag:

$$[
y_{t-7}
]$$

* Used to study repeating patterns

---

* **Autocorrelation**:

  * Time series correlated with its own past values
  * Strong autocorrelation observed here

---

* **Differencing**:

  * Subtracting previous values from current values

* 7-day differencing:

$$diff_7 = y_t - y_{t-7}$$

* Purpose:

  * Highlight deviations from weekly pattern
  * Reduce seasonality

---

* Observation:

  * Large deviations occurred during Memorial Day weekend
  * Holidays disrupt regular seasonal patterns

---

* **Mean Absolute Error (MAE)**:

  * Measures average absolute prediction error

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

* Naive forecasting results:

  * Bus:

    * ~43,916 riders error
  * Rail:

    * ~42,143 riders error

---

* **Mean Absolute Percentage Error (MAPE)**:

  * Error relative to actual values

$$MAPE = \frac{1}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

* Results:

  * Bus:

    * ~8.3%
  * Rail:

    * ~9.0%

* Insight:

  * MAE can be misleading when scales differ
  * MAPE normalizes errors relative to target size

---

* Common forecasting metrics:

  * MAE
  * MAPE
  * MSE

---

* **Mean Squared Error (MSE)**:

  * Penalizes large errors more strongly

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

* Useful when:

  * Large errors are especially costly

---

* **Monthly resampling**:

  * Convert daily data → monthly averages

* **Rolling average / moving average**:

  * Smooths noise
  * Reveals long-term trends

* 12-month rolling average:

  * Used to visualize:

    * yearly seasonality
    * long-term trend

---

* **Yearly seasonality**:

  * Patterns repeating every year
  * More visible in rail ridership

---

* **12-month differencing**:

$$diff_{12} = y_t - y_{t-12}$$

* Effects:

  * Removes yearly seasonality
  * Removes long-term trend

---

* **Trend**:

  * Long-term increase or decrease in data

* Example:

  * Ridership decline from 2016–2019

---

* **Stationary time series**:

  * Statistical properties remain constant over time
  * No strong:

    * trend
    * seasonality

* Important because:

  * Easier to model and forecast

---

* Differencing helps create stationarity by removing:

  * trends
  * seasonal patterns

---

* Forecast reconstruction:

  * After forecasting differenced values:

    * Add back removed past values to recover original scale

---

* Important forecasting insight:

  * Short-term patterns often dominate next-day prediction
  * But long-term trends can still slightly improve accuracy

---

* Example trend adjustment:

  * If weekly ridership decreases by 570:

$$[
\hat{y}*t = y*{t-7} - 570
]$$

* This incorporates trend into naive forecast

---

* Major concepts introduced:

  * Time series
  * Multivariate vs univariate
  * Forecasting
  * Seasonality
  * Trend
  * Lag
  * Autocorrelation
  * Differencing
  * Moving averages
  * Stationarity
  * MAE / MAPE / MSE


In [ ]:
filepath = tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir=".",
    extract=True
)
if "_extracted" in filepath:
    ridership_path = Path(filepath) / "ridership"
else:
    ridership_path = Path(filepath).with_name("ridership")

In [ ]:
import pandas as pd
from pathlib import Path

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [ ]:
df.head()

In [ ]:
import matplotlib.pyplot as plt

df["2019-03":"2019-05"].plot(grid=True, marker=".", figsize=(8, 3.5))
save_fig("daily_ridership_plot")  # extra code – saves the figure for the book
plt.show()

In [ ]:
diff_7 = df[["bus", "rail"]].diff(7)["2019-03":"2019-05"]

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
df.plot(ax=axs[0], legend=False, marker=".")  # original time series
df.shift(7).plot(ax=axs[0], grid=True, legend=False, linestyle=":")  # lagged
diff_7.plot(ax=axs[1], grid=True, marker=".")  # 7-day difference time series
axs[0].set_ylim([170_000, 900_000])  # extra code – beautifies the plot
save_fig("differencing_plot")  # extra code – saves the figure for the book
plt.show()

In [ ]:
list(df.loc["2019-05-25":"2019-05-27"]["day_type"])

In [ ]:
diff_7.abs().mean()

In [ ]:
targets = df[["bus", "rail"]]["2019-03":"2019-05"]
(diff_7 / targets).abs().mean()

In [ ]:
period = slice("2001", "2019")
try:
    df_monthly = df.select_dtypes(include="number").resample('ME').mean()  # compute the mean for each month
    rolling_average_12_months = df_monthly.loc[period].rolling(window=12).mean()
except ValueError as ex:
    try:
        df_monthly = df.select_dtypes(include="number").resample('M').mean()  # compute the mean for each month
        rolling_average_12_months = df_monthly.loc[period].rolling(window=12).mean()
    except ValueError as ex:
        df_monthly = df.resample('M').mean()  # compute the mean for each month
        rolling_average_12_months = df_monthly[period].rolling(window=12).mean()

fig, ax = plt.subplots(figsize=(8, 4))
df_monthly[period].plot(ax=ax, marker=".")
rolling_average_12_months.plot(ax=ax, grid=True, legend=False)
save_fig("long_term_ridership_plot")  # extra code – saves the figure for the book
plt.show()

In [ ]:
df_monthly.diff(12)[period].plot(grid=True, marker=".", figsize=(8, 3))
save_fig("yearly_diff_plot")  # extra code – saves the figure for the book
plt.show()

##The ARMA Model Family

* **ARMA (AutoRegressive Moving Average)**:

  * Classical statistical model for time series forecasting
  * Developed by Herman Wold in the 1930s
  * Combines:

    * Autoregression (AR)
    * Moving Average (MA)

---

* **ARMA forecasting equation**:

$$\hat{y}^{(t)} = \sum_{i=1}^{p} \alpha_i y^{(t-i)} + \sum_{i=1}^{q} \theta_i \epsilon^{(t-i)}$$

* Forecast error:

$$\epsilon^{(t)} = y^{(t)} - \hat{y}^{(t)}$$

---

* Components of ARMA:

  * $( \hat{y}^{(t)} ):$

    * Predicted value at time ( t )

  * $( y^{(t)} )$:

    * Actual value at time ( t )

  * $( \epsilon^{(t)} ):$

    * Forecast error (residual)

---

* **Autoregressive (AR) component**:

  * Uses past values to predict future

$$\sum_{i=1}^{p} \alpha_i y^{(t-i)}$$

* Key concepts:

  * ( p ):

    * Number of past steps considered
    * Hyperparameter
  * ( $\alpha_i$ ):

    * Learned weights

* Meaning:

  * Prediction based on previous observations

---

* **Moving Average (MA) component**:

  * Uses past forecast errors to improve prediction

$$\sum_{i=1}^{q} \theta_i \epsilon^{(t-i)}$$

* Key concepts:

  * ( q ):

    * Number of previous errors used
    * Hyperparameter
  * ($ \theta_i $):

    * Learned weights

* Meaning:

  * Corrects prediction using recent mistakes

---

* **Core intuition of ARMA**:

  * Forecast =

    * weighted past values
    * * correction from past errors

---

* **Important assumption**:

  * ARMA assumes the time series is **stationary**

* **Stationary time series**:

  * Statistical properties remain stable over time
  * No strong:

    * trend
    * changing variance
    * seasonality

---

* **Differencing**:

  * Used to make series stationary

* One-step differencing:

$$y'*t = y_t - y*{t-1}$$

* Acts like:

  * Approximation of derivative/slope

---

* **Effect on linear trends**:

  * Removes linear trend

* Example:

Original:
$[
[3,5,7,9,11]
]$

After differencing:
$[
[2,2,2,2]
]$

* Constant slope becomes constant values

---

* **Quadratic trends**:

  * One differencing not enough

* Example:

Original:

[1,4,9,16,25,36]


1st differencing:

[3,5,7,9,11]


2nd differencing:

[2,2,2,2]


* Second differencing removes quadratic trend

---

* **Order of differencing**:

  * Applying differencing ( d ) times:

    * Approximates ( $d^{th}$ ) derivative

---

* **Order of integration (( d ))**:

  * Hyperparameter representing:

    * Number of differencing rounds

* Effects:

  * ( d=1 ):

    * removes linear trend
  * ( d=2 ):

    * removes quadratic trend
  * General:

    * removes polynomial trends up to degree ( d )

---

* **Important connection**:

  * ARMA + differencing →

    * leads toward ARIMA models later

---

* **Core takeaway**:

  * ARMA forecasts using:

    * past observations
    * past errors
  * Works best on stationary data
  * Differencing is crucial for removing trends and stabilizing time series


* **ARIMA (AutoRegressive Integrated Moving Average)**:

  * Extension of ARMA model
  * Introduced in 1970 by:

    * George Box
    * Gwilym Jenkins

* ARIMA adds:

  * **Differencing** capability to ARMA

---

* **Meaning of ARIMA components**:

  * **AR** → AutoRegressive
  * **I** → Integrated
  * **MA** → Moving Average

---

* **Integrated (I) part**:

  * Refers to:

    * applying differencing ( d ) times

* Purpose:

  * Transform non-stationary series → stationary series

---

* **ARIMA workflow**:

1. Apply differencing ( d ) times

2. Convert series into a more stationary form

3. Apply ARMA model on transformed data

4. Forecast future differenced values

5. Reverse differencing by adding back removed terms

---

* **Differencing operation**:

First-order differencing:

$$y'*t = y_t - y*{t-1}$$

Second-order differencing:

$$y''_t = y'*t - y'*{t-1}$$

---

* **ARIMA notation**:

ARIMA(p,d,q)

* Parameters:

  * ( p ):

    * autoregressive order
  * ( d ):

    * differencing order (integration order)
  * ( q ):

    * moving average order

---

* **Key idea**:

  * ARMA alone assumes:

    * stationary data
  * ARIMA handles:

    * non-stationary data
    * trends
    * some seasonal effects (after differencing)

---

* **Forecast reconstruction**:

  * After predicting differenced values:

    * add back previously removed values
  * Restores predictions to original scale

---

* **Core intuition**:

  * ARIMA =

$$[
\text{Differencing} + \text{ARMA}
]$$

---

* **Why ARIMA became important**:

  * Many real-world time series:

    * contain trends
    * are non-stationary
  * ARIMA makes classical forecasting practical for such data

---

* **Core takeaway**:

  * ARIMA improves ARMA by introducing differencing
  * Makes it possible to model:

    * trending
    * non-stationary time series effectively


* **SARIMA (Seasonal ARIMA)**:

  * Extension of ARIMA
  * Designed for time series with:

    * strong seasonal patterns

* Example seasonal frequencies:

  * Weekly
  * Monthly
  * Yearly

---

* **Key idea**:

  * SARIMA models:

    1. Normal trend/non-seasonal behavior
    2. Seasonal repeating behavior

---

* **SARIMA builds on ARIMA**:

  * Uses standard ARIMA for:

    * short-term dynamics
  * Adds another ARIMA-like component for:

    * seasonal structure

---

* **SARIMA notation**:

$SARIMA(p,d,q)(P,D,Q)_s$

---

* **Non-seasonal parameters**:

  * Same as ARIMA

  * ( p ):

    * autoregressive order

  * ( d ):

    * differencing order

  * ( q ):

    * moving average order

---

* **Seasonal parameters**:

  * ( P ):

    * seasonal autoregressive order

  * ( D ):

    * seasonal differencing order

  * ( Q ):

    * seasonal moving average order

  * ( s ):

    * seasonal period length

---

* **Meaning of seasonal period ( s )**:

  * Number of time steps in one season

* Examples:

  * Daily data with weekly seasonality: s = 7


* Monthly data with yearly seasonality: s = 12



---

* **Seasonal autoregression (P)**:

  * Uses lagged seasonal values:

$$[
t-s,\ t-2s,\ t-3s,\dots
]$$

* Example:

  * Weekly seasonality:

$$[
t-7,\ t-14,\ t-21
]$$

---

* **Seasonal differencing (D)**:

  * Removes seasonal trends

* Seasonal differencing formula:

$y'*t = y_t - y*{t-s}$

* Example:

  * Weekly differencing:

$$[
y_t - y_{t-7}
]$$

---

* **Seasonal moving average (Q)**:

  * Uses past seasonal forecast errors:

$$[
\epsilon_{t-s},\ \epsilon_{t-2s},\dots
]$$

---

* **Total hyperparameters in SARIMA**:

  * 7 parameters:

$$[
(p,d,q,P,D,Q,s)
]$$

---

* **Core intuition**:

  * ARIMA handles:

    * trend + short-term structure
  * SARIMA additionally handles:

    * repeating seasonal cycles

---

* **Practical example**:

  * Transit ridership with weekly repetition:

    * SARIMA can directly model:

$$[
t-7,\ t-14,\ t-21
]$$

* Much better than plain ARIMA for seasonal data

---

* **Core takeaway**:

  * SARIMA is powerful for:

    * seasonal time series forecasting
  * Combines:

    * trend modeling
    * differencing
    * autoregression
    * moving average
    * seasonal dynamics all together


* Goal:

  * Fit a **SARIMA model** on rail ridership data
  * Forecast next day ridership

---

* Library used:

  * `statsmodels`
  * Contains:

    * ARIMA
    * SARIMA
    * Other statistical time series models

---

* Import statement:

```python
from statsmodels.tsa.arima.model import ARIMA
```

---

* Data preparation:

  * Use rail ridership data only
  * Select data:

$$[
2019\text{-}01\text{-}01 \rightarrow 2019\text{-}05\text{-}31
]$$

* Set frequency:

```python
.asfreq("D")
```

* Purpose:

  * Explicitly define daily frequency
  * Avoid warning from statsmodels
  * Does not change data here

---

* SARIMA model creation:

```python
model = ARIMA(
    rail_series,
    order=(1,0,0),
    seasonal_order=(0,1,1,7)
)
```

---

* Non-seasonal parameters:

(p,d,q) = (1,0,0)

* Meaning:

  * ( p = 1 ):

    * use 1 past value
  * ( d = 0 ):

    * no regular differencing
  * ( q = 0 ):

    * no moving average component

---

* Seasonal parameters:

(P,D,Q,s) = (0,1,1,7)

* Meaning:

  * ( P = 0 ):

    * no seasonal autoregression
  * ( D = 1 ):

    * one seasonal differencing step
  * ( Q = 1 ):

    * one seasonal moving average term
  * ( s = 7 ):

    * weekly seasonality

---

* Model training:

```python
model = model.fit()
```

* Important API difference:

  * In `statsmodels`:

    * data passed during model creation
  * Unlike Scikit-Learn:

    * where data usually passed to `fit()`

---

* Forecasting:

```python
y_pred = model.forecast()
```

* Forecast result:

[
427,758.6
]

* Actual ridership:

[
379,044
]

---

* Forecast error analysis:

  * Error:

[
12.9%
]

* Worse than naive forecasting:

  * Naive prediction:

[
426,932
]

* Naive error:

[
12.6%
]

---

* Important lesson:

  * Single forecast evaluation can be misleading
  * Need evaluation across many days

---

* Rolling forecast evaluation:

  * Train and forecast repeatedly for:

$$[
2019\text{-}03\text{-}01 \rightarrow 2019\text{-}05\text{-}31
]$$

* Process:

  1. Train model using data up to current day
  2. Forecast next day
  3. Repeat daily

---

* Important concept:

  * Model retrained every day

```python
model.fit()
```

* Why:

  * Incorporate newest available data

---

* MAE evaluation:

```python
mae = (y_preds - actual_values).abs().mean()
```

* Result:

$$[
MAE \approx 32,041
]$$

---

* Comparison with naive forecasting:

| Model             | MAE     |
| ----------------- | ------- |
| Naive Forecasting | ~42,143 |
| SARIMA            | ~32,041 |

* SARIMA performs significantly better overall

---

* Key insight:

  * A model can fail badly on one example
  * Yet still perform better on average

---

* Hyperparameter tuning:

  * Usually done using:

    * Grid Search (brute-force search)

---

* Typical hyperparameter ranges:

* Non-seasonal:

  * ( p,q ):

[
0 \rightarrow 2
]

* sometimes:

$$[
5 \text{ or } 6
]$$

* Differencing:

  * ( d,D ):

[
0,1
]

* sometimes:

[
2
]

* Seasonal period:

  * ( s ):

    * determined from seasonality
    * here:

[
s = 7
]

---

* Model selection criterion:

  * Choose model with:

    * lowest MAE
    * or another business-relevant metric

---

* Important forecasting workflow:

  1. Analyze seasonality/trend
  2. Choose SARIMA structure
  3. Train on historical data
  4. Forecast future values
  5. Evaluate with metrics
  6. Tune hyperparameters

---

* Core takeaway:

  * SARIMA can outperform naive forecasting significantly
  * Seasonal differencing is especially powerful for weekly patterns
  * Time series evaluation should be done over many rolling forecasts, not a single prediction


In [ ]:
if "google.colab" in sys.modules:
    %pip install -q -U statsmodels

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

origin, today = "2019-01-01", "2019-05-31"
rail_series = df.loc[origin:today]["rail"].asfreq("D")
model = ARIMA(rail_series,
              order=(1, 0, 0),
              seasonal_order=(0, 1, 1, 7))
model = model.fit()
y_pred = model.forecast()  # returns 427,758.6

In [ ]:
y_pred[0]  # ARIMA forecast

In [ ]:
df["rail"].loc["2019-06-01"]  # target value

In [ ]:
df["rail"].loc["2019-05-25"]  # naive forecast (value from one week earlier)

In [ ]:
origin, start_date, end_date = "2019-01-01", "2019-03-01", "2019-05-31"
time_period = pd.date_range(start_date, end_date)
rail_series = df.loc[origin:end_date]["rail"].asfreq("D")
y_preds = []
for today in time_period.shift(-1):
    model = ARIMA(rail_series[origin:today],  # train on data up to "today"
                  order=(1, 0, 0),
                  seasonal_order=(0, 1, 1, 7))
    model = model.fit()  # note that we retrain the model every day!
    y_pred = model.forecast().iloc[0]
    y_preds.append(y_pred)

y_preds = pd.Series(y_preds, index=time_period)
mae = (y_preds - rail_series[time_period]).abs().mean()  # returns 32,040.7

In [ ]:
mae

In [ ]:
# extra code – displays the SARIMA forecasts
fig, ax = plt.subplots(figsize=(8, 3))
rail_series.loc[time_period].plot(label="True", ax=ax, marker=".", grid=True)
ax.plot(y_preds, color="r", marker=".", label="SARIMA Forecasts")
plt.legend()
plt.show()

In [ ]:
# extra code – shows how to plot the Autocorrelation Function (ACF) and the
#              Partial Autocorrelation Function (PACF)

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))
plot_acf(df[period]["rail"], ax=axs[0], lags=35)
axs[0].grid()
plot_pacf(df[period]["rail"], ax=axs[1], lags=35, method="ywm")
axs[1].grid()
plt.show()

## Preparing the Data for Machine Learning Models

* After establishing baselines:

  * Naive Forecasting
  * SARIMA
  * Next step:

    * Apply machine learning models for forecasting

---

* Forecasting goal:

  * Predict tomorrow’s ridership using:

$$[
56 \text{ previous days}
]$$

* 56 days = 8 weeks

---

* Input-output structure:

  * Input:

    * sequence from:

$$[
t-55 \rightarrow t
]$$

```
- total length:
  
```

$$[
56
]$$

* Target:

$$[
t+1
]$$

* This is:

  * **sequence-to-vector forecasting**

---

* Key training trick:

  * Use **sliding windows**

* Every past 56-day window becomes:

  * one training sample

* Target:

  * value immediately after window

---

* Example windowing:

Input:
$$[
[x_{t-55}, x_{t-54}, ..., x_t]
]$$

Target:
$$[
x_{t+1}
]$$

---

* Keras utility function:

  * `timeseries_dataset_from_array()`

* Purpose:

  * Automatically creates:

    * sliding windows
    * corresponding targets
    * `tf.data.Dataset`

---

* Example:

Input series:
$$[
[0,1,2,3,4,5]
]$$

Window length:
[$$
3
]$$

Generated windows + targets:

| Window  | Target |
| ------- | ------ |
| [0,1,2] | 3      |
| [1,2,3] | 4      |
| [2,3,4] | 5      |

---

* Important concept:

  * Targets shifted forward relative to inputs

---

* Batch behavior:

  * Batch size = 2
  * Last batch may contain fewer samples if:

$$[
\text{dataset size} \not\equiv 0 \pmod{\text{batch size}}
]$$

---

* Alternative method:

  * Using `tf.data.Dataset.window()`

* Gives:

  * More control over preprocessing
  * More flexible than helper utility

---

* `window()` behavior:

  * Produces:

    * dataset of datasets (nested dataset)

* Example:

Windows:
$$[
[0,1,2,3]
]$$
$$[
[1,2,3,4]
]$$
$$[
[2,3,4,5]
]$$

---

* `shift=1`:

  * Sliding window moves:

$$[
1 \text{ step at a time}
]$$

---

* Problem:

  * Final windows may be smaller near sequence end

* Solution:

  * Use:

```python id="0gok77"
drop_remainder=True
```

* Removes incomplete windows

---

* Nested dataset issue:

  * Models expect:

    * tensors
  * not:

    * datasets

---

* `flat_map()`:

  * Flattens nested datasets

* Concept:

Nested:
$$[
{{1,2},{3,4,5,6}}
]$$

Flattened:
$$[
{1,2,3,4,5,6}
]$$

---

* `flat_map()` with transformation:

  * Can modify windows before flattening

* Example:

```python id="30c6ik"
lambda ds: ds.batch(2)
```

* Produces batched tensors

---

* Window-to-tensor conversion:

```python id="9m0t2t"
window_dataset.batch(length)
```

* Converts one window dataset → one tensor

---

* Helper function:

```python id="1b7w4q"
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))
```

* Purpose:

  * Simplify sliding window creation

---

* Splitting windows into:

  * inputs
  * targets

Using:

```python id="vrlz0g"
window[:-1]
window[-1]
```

---

* Example:

Window:
[
[0,1,2,3]
]

Input:
[
[0,1,2]
]

Target:
[
3
]

---

* Time series train/validation/test split:

  * Must split across **time**
  * Never random split

---

* Dataset split:

Training:
$$[
2016 \rightarrow 2018
]$$

Validation:
$$[
2019\text{-}01 \rightarrow 2019\text{-}05
]$$

Test:
$$[
2019\text{-}06 \rightarrow \text{future}
]$$

---

* Scaling:

  * Divide by:

$$[
10^6
]$$

* Purpose:

  * Keep values near:

$$[
0 \rightarrow 1
]$$

* Benefits:

  * Better weight initialization
  * More stable learning
  * Better gradient behavior

---

* Important ML assumption:

  * Gradient descent assumes:

    * IID data

      * Independent and Identically Distributed

---

* Why shuffle training windows:

  * Prevent correlated training order
  * Improve gradient descent learning

* Important:

  * Shuffle:

    * windows
  * NOT:

    * elements inside windows

---

* Training dataset creation:

Parameters:

* sequence length:

[
56
]

* batch size:

[
32
]

* shuffle:

```python id="grutq8"
shuffle=True
```

---

* Validation dataset:

  * No shuffling

* Reason:

  * Evaluation should preserve natural order

---

* Core concepts introduced:

  * Sliding windows
  * Sequence forecasting
  * tf.data pipeline
  * Window datasets
  * flat_map()
  * map()
  * Batching
  * Time-based splitting
  * Scaling
  * IID assumption
  * Training/validation/test workflow

---

* Core takeaway:

  * Time series forecasting in ML converts sequential data into:

$$[
(\text{past window}) \rightarrow (\text{future target})
]$$

* Windowing is one of the most important preprocessing steps in sequence modeling


In [ ]:
import tensorflow as tf

my_series = [0, 1, 2, 3, 4, 5]
my_dataset = tf.keras.utils.timeseries_dataset_from_array(
    my_series,
    targets=my_series[3:],  # the targets are 3 steps into the future
    sequence_length=3,
    batch_size=2
)
list(my_dataset)

In [ ]:
for window_dataset in tf.data.Dataset.range(6).window(4, shift=1):
    for element in window_dataset:
        print(f"{element}", end=" ")
    print()

In [ ]:
dataset = tf.data.Dataset.range(6).window(4, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window_dataset: window_dataset.batch(4))
for window_tensor in dataset:
    print(f"{window_tensor}")

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))

In [ ]:
dataset = to_windows(tf.data.Dataset.range(6), 4)
dataset = dataset.map(lambda window: (window[:-1], window[-1]))
list(dataset.batch(2))

In [ ]:
rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

In [ ]:
seq_length = 56
tf.random.set_seed(42)  # extra code – ensures reproducibility
train_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_train.to_numpy(),
    targets=rail_train[seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_valid.to_numpy(),
    targets=rail_valid[seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

## Forecasting Using a Linear Model

* First ML model:

  * Basic **linear regression model** using Keras

---

* Model architecture:

```python id="j7sy4r"
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=[seq_length])
])
```

* Structure:

  * Single Dense layer
  * Output: 1

* Input size: 56

* Meaning:

  * Model receives:

    * 56 past ridership values
  * Predicts:

    * next day's ridership

---

* Important observation:

  * This is NOT an RNN
  * It treats:

    * input as plain vector
  * No temporal memory/state

---

* Linear model equation:

$$\hat{y} = w_1x_1 + w_2x_2 + \cdots + w_{56}x_{56} + b$$

* Each past day gets:

  * learned weight

---

* Loss function:

  * **Huber Loss**

```python id="ay8g8i"
loss=tf.keras.losses.Huber()
```

---

* Why Huber loss:

  * More robust than MSE for outliers
  * Smoother than MAE
  * Often easier to optimize

---

* Huber loss behavior:

  * Small errors:

    * behaves like MSE
  * Large errors:

    * behaves like MAE

---

* Optimizer used:

  * SGD with momentum

```python id="48z4i4"
SGD(learning_rate=0.02, momentum=0.9)
```

---

* Hyperparameters:

  * Learning rate:

[
0.02
]

* Momentum:

[
0.9
]

---

* Momentum:

  * Helps accelerate optimization
  * Reduces oscillations
  * Uses past gradient direction

---

* Metric used:

  * MAE

```python id="o9f2hj"
metrics=["mae"]
```

* Easier to interpret for forecasting

---

* Early stopping:

```python id="0tmhmy"
EarlyStopping(
    monitor="val_mae",
    patience=50,
    restore_best_weights=True
)
```

---

* Purpose of early stopping:

  * Prevent overfitting
  * Stop training when validation performance stops improving

---

* Parameters:

  * `monitor="val_mae"`

    * track validation MAE

  * `patience=50`

    * wait 50 epochs before stopping

  * `restore_best_weights=True`

    * restore best-performing model weights

---

* Training:

```python id="nqcg59"
model.fit(...)
```

* Maximum epochs:

[
500
]

* But early stopping may stop earlier

---

* Validation result:

[
MAE \approx 37,866
]

---

* Performance comparison:

| Model             | MAE     |
| ----------------- | ------- |
| Naive Forecasting | ~42,143 |
| Linear Model      | ~37,866 |
| SARIMA            | ~32,041 |

---

* Key insight:

  * Even simple linear models can outperform naive forecasting
  * But SARIMA still performs better because:

    * explicitly models seasonality + temporal structure

---

* Limitation of linear model:

  * No sequential memory
  * No recurrent connections
  * Cannot model temporal dynamics deeply

---

* Important transition:

  * Next step:

    * use RNNs
  * Goal:

    * capture sequential dependencies better than linear models

---

* Core takeaway:

  * Sliding-window forecasting can already work with simple dense models
  * But temporal models like SARIMA and RNNs are better suited for sequence patterns


In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=[seq_length])
])
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae", patience=50, restore_best_weights=True)
opt = tf.keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)
model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
history = model.fit(train_ds, validation_data=valid_ds, epochs=500,
                    callbacks=[early_stopping_cb])

In [ ]:
# extra code – evaluates the model
valid_loss, valid_mae = model.evaluate(valid_ds)
valid_mae * 1e6

## Forecasting Using a Linear Model

* First real RNN model:

  * Uses a `SimpleRNN` layer
  * Designed specifically for sequential/time-series data

* Key difference from dense model:

  * Dense model:

    * sees all 56 values at once
    * no memory
  * RNN:

    * processes values step-by-step
    * carries information forward using hidden state (memory)

---

* RNN input format:

$$[
[\text{batch size}, \text{time steps}, \text{features}]
]$$

* Here:

  * Time steps:

[
56
]

* Features:

[
1
]

* Because rail ridership is a univariate time series

---

* Hidden state:

  * Core memory mechanism of RNN

* At each step:

$$h_t = f(x_t, h_{t-1})$$

* Meaning:

  * Current state depends on:

    * current input
    * previous memory

---

* First RNN model failure:

  * Used only:

[
1
]

recurrent neuron

* Problems:

  1. Extremely tiny memory
  2. Only 3 trainable parameters
  3. Could not capture complex weekly patterns

---

* Another major problem:

  * Default activation:

$$[
\tanh
]$$

* Output range:

$$-1 \leq \tanh(x) \leq 1$$

* But normalized ridership values reached:

[
1.4
]

* So model physically could not predict larger values

---

* Important lesson:

  * Activation functions can restrict prediction range

---

* Improved model:

  * Increased recurrent neurons:

$$[
1 \rightarrow 32
]$$

* Effects:

  * Much larger memory capacity
  * Can store richer temporal patterns
  * Hidden state becomes 32-dimensional instead of scalar

---

* Dense output layer added:

  * Converts:

$$[
32 \text{-dimensional memory} \rightarrow 1 \text{ prediction}
]$$

* No activation used:

  * Output unrestricted

---

* Important Keras behavior:

  * By default RNN returns:

    * only final time-step output

* This makes it:

  * sequence-to-vector architecture

---

* Validation performance:

| Model             | MAE    |
| ----------------- | ------ |
| Naive Forecasting | ~42k   |
| Linear Model      | ~38k   |
| SARIMA            | ~32k   |
| Improved RNN      | ~27.7k |

---

* Why improved RNN works better:

  * Learns temporal dependencies automatically
  * Maintains sequential memory
  * Better at capturing repeating weekly structures

---

* Very important insight:

  * RNN performed well even without:

    * differencing
    * trend removal
    * stationarity preprocessing

* Unlike ARIMA/SARIMA:

  * RNNs can often learn these patterns directly

---

* Core concepts introduced:

  * Sequential processing
  * Hidden state / memory
  * Recurrent computation
  * Temporal dependency learning
  * Sequence-to-vector forecasting
  * Memory capacity importance
  * Activation output constraints
  * Hidden representation dimensionality

---

* Biggest conceptual takeaway:

  * Dense networks look at history as static numbers
  * RNNs treat history as evolving temporal information over time


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

In [ ]:
# extra code – defines a utility function we'll reuse several time

def fit_and_evaluate(model, train_set, valid_set, learning_rate, epochs=500):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=50, restore_best_weights=True)
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae * 1e6

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
univar_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, input_shape=[None, 1]),
    tf.keras.layers.Dense(1)  # no activation function by default
])

In [ ]:
# extra code – compiles, fits, and evaluates the model, like earlier
fit_and_evaluate(univar_model, train_ds, valid_ds, learning_rate=0.05)

## Forecasting Using a Deep RNN

* **Deep RNN**:

  * Created by stacking multiple recurrent layers
  * Similar idea to deep feedforward neural networks:

    * deeper hierarchy
    * richer feature extraction

---

* Basic intuition:

  * Lower RNN layers:

    * learn simple temporal patterns
  * Higher RNN layers:

    * learn more abstract/high-level sequence patterns

---

* Architecture used:

  * 3 stacked `SimpleRNN` layers
  * Final Dense layer for prediction

---

* First two RNN layers:

  * sequence-to-sequence

* Meaning:

  * Output generated at every time step

---

* Last RNN layer:

  * sequence-to-vector

* Meaning:

  * Only final time-step output returned

---

* Final Dense layer:

  * Converts hidden representation → final forecast value

---

* Very important parameter:

```python id="ry2o0g"
return_sequences=True
```

---

* Why it matters:

  * Recurrent layers expect:

$$[
3D \text{ sequence input}
]$$

* Without `return_sequences=True`:

  * layer outputs only final timestep
  * output becomes:

[
2D
]

instead of:

[
3D
]

* Then next RNN layer cannot process sequence properly

---

* Rule to remember:

  * All recurrent layers except last:

```python id="m8nccr"
return_sequences=True
```

* Last recurrent layer:

  * optional depending on task

---

* Shape flow intuition:

With `return_sequences=True`:

$$[
(batch,\ time,\ features)
]$$

Without it:

$$[
(batch,\ features)
]$$

---

* Deep RNN information flow:

  * First layer processes raw sequence
  * Second layer processes temporal features from first layer
  * Third layer builds even deeper temporal representation

---

* Important concept:

  * Stacked RNNs increase:

    * representational power
    * learning capacity
    * abstraction capability

---

* Validation MAE:

$$[
\approx 31,211
]$$

---

* Performance comparison:

| Model                       | MAE    |
| --------------------------- | ------ |
| Naive Forecasting           | ~42k   |
| Linear Model                | ~38k   |
| SARIMA                      | ~32k   |
| Single-layer RNN (32 units) | ~27.7k |
| Deep RNN                    | ~31.2k |

---

* Important insight:

  * Deeper model performed worse than simpler RNN

---

* Why deeper RNN underperformed:

  * Model too large for task
  * Possible overfitting
  * Added complexity without enough benefit

---

* Very important ML lesson:

  * Bigger/deeper ≠ automatically better
  * Model complexity must match:

    * dataset size
    * task difficulty
    * pattern complexity

---

* Another important RNN insight:

  * Deeper RNNs are harder to train because of:

    * unstable gradients
    * long dependency chains
    * optimization difficulty

---

* Core concepts introduced:

  * Deep RNN
  * Stacked recurrent layers
  * Sequence-to-sequence vs sequence-to-vector
  * `return_sequences`
  * 3D tensor requirement
  * Hierarchical temporal feature learning
  * Overfitting from excessive complexity

---

* Biggest conceptual takeaway:

  * Deep RNNs learn temporal features layer-by-layer, but increasing depth only helps when the task actually requires that extra complexity


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
deep_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, return_sequences=True, input_shape=[None, 1]),
    tf.keras.layers.SimpleRNN(32, return_sequences=True),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(1)
])

In [ ]:
# extra code – compiles, fits, and evaluates the model, like earlier
fit_and_evaluate(deep_model, train_ds, valid_ds, learning_rate=0.01)

## Forecasting Multivariate Time Series

* One major strength of neural networks:

  * Extremely flexible with input features
  * Can handle multivariate time series with minimal architecture changes

---

* Earlier model:

  * Used only:

$$[
\text{rail ridership}
]$$

* New idea:

  * Use additional related information:

    * bus ridership
    * rail ridership
    * tomorrow’s day type

---

* Important forecasting insight:

  * Future-known variables can be used

* Example:

  * Tomorrow’s:

    * weekday
    * weekend
    * holiday

is already known beforehand

---

* Feature engineering step:

  * Shift day type one day ahead

* Purpose:

  * Give model tomorrow’s calendar information during prediction

---

* One-hot encoding:

  * Day types:

$$[
W,\ A,\ U
]$$

converted into binary vectors

---

* One-hot encoding concept:

Example:

$$[
W = [1,0,0]
]$$

$$[
A = [0,1,0]
]$$

$$[
U = [0,0,1]
]$$

---

* Final dataset columns:

  * bus
  * rail
  * next_day_type_W
  * next_day_type_A
  * next_day_type_U

Total features per timestep:

[
5
]

---

* Important transition:

  * From:

    * univariate time series
  * To:

    * multivariate time series

---

* Multivariate RNN input shape:

$$[
[\text{batch},\ \text{time steps},\ 5]
]$$

* At each timestep:

  * model receives 5 features instead of 1

---

* Important insight:

  * Architecture barely changes
  * Only input dimensionality changes

---

* Target:

  * Still forecasting only:

$$[
\text{rail ridership}
]$$

---

* Model architecture remains simple:

  * SimpleRNN(32)
  * Dense(1)

* Only difference:

```python id="95b49x"
input_shape=[None, 5]
```

---

* Why performance improves:

  * Model now sees correlated signals

Example:

* Bus and rail ridership often move together
* Holidays affect both systems similarly

---

* Validation MAE improvement:

| Model             | MAE    |
| ----------------- | ------ |
| Naive Forecasting | ~42k   |
| Linear Model      | ~38k   |
| SARIMA            | ~32k   |
| Univariate RNN    | ~27.7k |
| Multivariate RNN  | ~22.1k |

---

* Very important ML principle:

  * Better features often matter more than deeper models

---

* Additional idea:

  * Forecast both:

    * bus
    * rail

simultaneously

---

* Multi-output forecasting:

  * Output layer:

[
Dense(2)
]

instead of:

[
Dense(1)
]

---

* Targets become:

$$[
[\text{bus},\ \text{rail}]
]$$

---

* Important ML concept:

  * Multi-task learning

* One model learns multiple related tasks together

---

* Why multitask learning can help:

  1. Shared features between tasks
  2. Better generalization
  3. Acts as regularization
  4. Reduces overfitting

---

* Example shared patterns:

  * Holidays affect both bus and rail
  * Weekly seasonality shared
  * Long-term ridership trends shared

---

* But important caveat:

  * Multitask learning is not always better

* In this case:

  * Dedicated single-task models performed slightly better

---

* Multitask model results:

  * Rail MAE:

[
25,330
]

* Bus MAE:

[
26,369
]

---

* Important practical insight:

  * Adding relevant contextual features can dramatically improve forecasting performance

* Especially:

  * calendar information
  * correlated time series
  * external signals

---

* Core concepts introduced:

  * Multivariate time series
  * Feature engineering
  * Future-known features
  * One-hot encoding
  * Multi-feature RNN input
  * Multi-output forecasting
  * Multi-task learning
  * Shared representations
  * Regularization through multitask learning

---

* Biggest conceptual takeaway:

  * RNN performance depends not only on architecture, but heavily on the quality and richness of temporal/contextual features provided to the model


In [ ]:
df_mulvar = df[["bus", "rail"]] / 1e6  # use both bus & rail series as input
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # we know tomorrow's type
df_mulvar = pd.get_dummies(df_mulvar, dtype=float)  # one-hot encode the day type

In [ ]:
mulvar_train = df_mulvar["2016-01":"2018-12"]
mulvar_valid = df_mulvar["2019-01":"2019-05"]
mulvar_test = df_mulvar["2019-06":]

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility

train_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_train.to_numpy(),  # use all 5 columns as input
    targets=mulvar_train["rail"][seq_length:],  # forecast only the rail series
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_valid.to_numpy(),
    targets=mulvar_valid["rail"][seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
mulvar_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, input_shape=[None, 5]),
    tf.keras.layers.Dense(1)
])

In [ ]:
# extra code – compiles, fits, and evaluates the model, like earlier
fit_and_evaluate(mulvar_model, train_mulvar_ds, valid_mulvar_ds, learning_rate=0.05)

In [ ]:
# extra code – build and train a multitask RNN that forecasts both bus and rail

tf.random.set_seed(42)

seq_length = 56
train_multask_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_train.to_numpy(),
    targets=mulvar_train[["bus", "rail"]][seq_length:],  # 2 targets per day
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_multask_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_valid.to_numpy(),
    targets=mulvar_valid[["bus", "rail"]][seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

tf.random.set_seed(42)
multask_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, input_shape=[None, 5]),
    tf.keras.layers.Dense(2)
])

fit_and_evaluate(multask_model, train_multask_ds, valid_multask_ds,
                 learning_rate=0.02)

In [ ]:
# extra code – evaluates the naive forecasts for bus
bus_naive = mulvar_valid["bus"].shift(7)[seq_length:]
bus_target = mulvar_valid["bus"][seq_length:]
(bus_target - bus_naive).abs().mean() * 1e6

In [ ]:
# extra code – evaluates the multitask RNN's forecasts both bus and rail
Y_preds_valid = multask_model.predict(valid_multask_ds)
for idx, name in enumerate(["bus", "rail"]):
    mae = 1e6 * tf.keras.metrics.MeanAbsoluteError()(
        mulvar_valid[name][seq_length:], Y_preds_valid[:, idx])
    print(name, int(mae))

## Forecasting Several Time Steps Ahead

* Earlier forecasting:

  * Predict only: t+1


(next timestep)

---

* Multi-step forecasting:

  * Predict several future values

Example:

* next 14 days

---

* Two main approaches for multi-step forecasting:

  1. Recursive forecasting
  2. Direct multi-output forecasting

---

* Approach 1: Recursive forecasting

  * Predict one step at a time
  * Feed prediction back into model
  * Repeat recursively

---

* Workflow:

  1. Predict next value
  2. Append prediction to sequence
  3. Use updated sequence to predict again
  4. Repeat until desired horizon reached

---

* Input shape remains:

$$[
[1,\ 56,\ 1]
]$$

* Meaning:

  * batch size:

[
1
]

* 56 timesteps
* 1 feature

---

* Key idea:

  * Model behaves as if its own predictions were real observations

---

* Important warning:

  * Errors accumulate over time

---

* Error accumulation problem:

  * Wrong prediction at step 1 affects:

    * step 2
    * step 3
    * step 4
    * etc.

* This causes:

  * drift
  * instability
  * degraded long-range forecasts

---

* Recursive forecasting works best for:

  * short forecasting horizons

---

* Core limitation:

  * Training distribution ≠ inference distribution

* During training:

  * model sees real historical data

* During prediction:

  * model sees its own imperfect outputs

---

* Approach 2: Direct multi-output forecasting

  * Predict all future steps simultaneously

---

* Instead of:

$$[
1 \text{ output}
]$$

* Model outputs:

$$[
14 \text{ outputs}
]$$

all at once

---

* Important dataset modification:

  * Targets become:

$$[
[t+1,\ t+2,\ \dots,\ t+14]
]$$

---

* Input-target structure:

Input:
$$[
[t-55,\dots,t]
]$$

Target:
$$[
[t+1,\dots,t+14]
]$$

---

* Dataset creation trick:

  * Create longer sequences:

[
56 + 14
]

* Then split:

  * first 56 → inputs
  * last 14 → targets

---

* Important function:

```python id="2fgkzn"
split_inputs_and_targets()
```

* Purpose:

  * Separate:

    * historical window
    * future prediction horizon

---

* Important concept:

  * Multivariate input still used
  * Forecasting only rail series

---

* Model architecture:

  * SimpleRNN(32)
  * Dense(14)

---

* Output layer meaning:

[
Dense(14)
]

* Predicts:

  * 14 future timesteps simultaneously

---

* Prediction shape:

[
[1,\ 14]
]

* Meaning:

  * 14 future forecast values

---

* Major advantage:

  * No recursive error accumulation

* Each future step predicted directly

---

* Tradeoff:

  * Longer horizon predictions still harder

* Usually:

  * Near-future predictions better
  * Far-future predictions less accurate

---

* Comparison of both approaches:

| Approach              | Strength | Weakness                |
| --------------------- | -------- | ----------------------- |
| Recursive forecasting | Simple   | Error accumulation      |
| Direct multi-output   | Stable   | Harder learning problem |

---

* Important forecasting insight:

  * Multi-output forecasting teaches model:

    * temporal future structure directly

---

* Transition to next concept:

  * Sequence-to-sequence (Seq2Seq) models

* Why:

  * Better suited for:

$$[
\text{sequence} \rightarrow \text{sequence}
]$$

forecasting tasks

---

* Core concepts introduced:

  * Multi-step forecasting
  * Recursive forecasting
  * Forecast horizon
  * Error accumulation
  * Direct forecasting
  * Multi-output prediction
  * Future sequence targets
  * Forecast stability

---

* Biggest conceptual takeaway:

  * Predicting many future steps is fundamentally harder because prediction errors can propagate through time, so architectures designed to model future sequences directly become increasingly important


In [ ]:
import numpy as np

X = rail_valid.to_numpy()[np.newaxis, :seq_length, np.newaxis]
for step_ahead in range(14):
    y_pred_one = univar_model.predict(X)
    X = np.concatenate([X, y_pred_one.reshape(1, 1, 1)], axis=1)

In [ ]:
# extra code – generates and saves Figure 15–11

# The forecasts start on 2019-02-26, as it is the 57th day of 2019, and they end
# on 2019-03-11. That's 14 days in total.
Y_pred = pd.Series(X[0, -14:, 0],
                   index=pd.date_range("2019-02-26", "2019-03-11"))

fig, ax = plt.subplots(figsize=(8, 3.5))
(rail_valid * 1e6)["2019-02-01":"2019-03-11"].plot(
    label="True", marker=".", ax=ax)
(Y_pred * 1e6).plot(
    label="Predictions", grid=True, marker="x", color="r", ax=ax)
ax.vlines("2019-02-25", 0, 1e6, color="k", linestyle="--", label="Today")
ax.set_ylim([200_000, 800_000])
plt.legend(loc="center left")
save_fig("forecast_ahead_plot")
plt.show()

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility

def split_inputs_and_targets(mulvar_series, ahead=14, target_col=1):
    return mulvar_series[:, :-ahead], mulvar_series[:, -ahead:, target_col]

ahead_train_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_train.to_numpy(),
    targets=None,
    sequence_length=seq_length + 14,
    batch_size=32,
    shuffle=True,
    seed=42
).map(split_inputs_and_targets)
ahead_valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    mulvar_valid.to_numpy(),
    targets=None,
    sequence_length=seq_length + 14,
    batch_size=32
).map(split_inputs_and_targets)

In [ ]:
tf.random.set_seed(42)

ahead_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

In [ ]:
# extra code – compiles, fits, and evaluates the model, like earlier
fit_and_evaluate(ahead_model, ahead_train_ds, ahead_valid_ds,
                 learning_rate=0.02)

In [ ]:
X = mulvar_valid.to_numpy()[np.newaxis, :seq_length]  # shape [1, 56, 5]
Y_pred = ahead_model.predict(X)  # shape [1, 14]

## Forecasting Using a Sequence-to-Sequence Model

* Previous model:

  * Forecasted future values only at:

$$[
\text{last timestep}
]$$

* New idea:

  * Forecast future values at:

$$[
\text{every timestep}
]$$

---

* Transition:

  * From:

    * sequence-to-vector
  * To:

    * sequence-to-sequence (Seq2Seq)

---

* Key Seq2Seq idea:

  * Every timestep produces future forecasts

---

* Example:

  * At timestep 0:

$$[
[t+1,\dots,t+14]
]$$

* At timestep 1:

$$[
[t+2,\dots,t+15]
]$$

* and so on...

---

* Very important benefit:

  * Much stronger gradient flow during training

---

* Why training improves:

  * Loss computed at every timestep
  * Not only final timestep

---

* Consequences:

  1. More gradient signals
  2. Faster learning
  3. More stable optimization
  4. Easier gradient propagation through time

---

* Important deep learning insight:

  * Long gradient paths are difficult for RNNs
  * Seq2Seq shortens effective gradient distance

---

* Target structure changes completely

Before:

* single vector target

Now:

* sequence of target vectors

---

* New target format:

Input:
$$[
[x_0,x_1,x_2,x_3]
]$$

Targets:
$$[
[[x_1,x_2],[x_2,x_3],[x_3,x_4],[x_4,x_5]]
]$$

---

* Meaning:

  * At every timestep:

    * predict future horizon

---

* Important conceptual point:

  * Targets may contain values appearing later in inputs

* But this is NOT cheating because:

  * RNNs are causal models

---

* Causal model:

  * Can only access:

$$[
\text{past and current information}
]$$

* Cannot see future timesteps during prediction

---

* Dataset preparation becomes more complex:

  * Need:

    * windows of windows

---

* Double windowing concept:

  1. Create future forecast windows
  2. Create sequences of those windows

---

* Result:

  * Input:

$$[
(\text{sequence length})
]$$

* Output:

$$[
(\text{sequence length} \times \text{forecast horizon})
]$$

---

* Seq2Seq model architecture:

* Important change:

```python id="akq8yx"
return_sequences=True
```

---

* Why crucial:

  * RNN now outputs:

    * hidden representation at every timestep

Instead of:

* only final timestep

---

* Dense layer behavior:

  * Automatically applied at each timestep

---

* Output structure:

  * Each timestep outputs:

[
14
]

future predictions

---

* Important equivalence:

  * Dense layer on sequences behaves similarly to:

```python id="ryvpxk"
Conv1D(kernel_size=1)
```

---

* Important Keras concept:

  * `TimeDistributed`
  * Applies same layer independently across timesteps

* But not needed here because:

  * Dense already supports sequence inputs

---

* Training:

  * Uses outputs from ALL timesteps

---

* Inference:

  * Usually only final timestep prediction matters

Example:

```python id="f03hz4"
predict(...)[0, -1]
```

* Meaning:

  * use last timestep’s forecast

---

* Performance trend:

  * Near future forecasts:

    * more accurate

* Distant future forecasts:

  * less accurate

---

* MAE progression:

  * ( t+1 ):

[
25,519
]

* ( t+2 ):

[
26,274
]

* ( t+14 ):

[
34,322
]

---

* Important forecasting insight:

  * Forecast uncertainty increases with horizon length

---

* Hybrid forecasting idea:

  * Combine:

    * direct forecasting
    * recursive forecasting

---

* Example:

  1. Predict next 14 days
  2. Append predictions
  3. Predict following 14 days
  4. Repeat

---

* Important limitation of SimpleRNNs:

  * Struggle with long sequences

Reasons:

* Vanishing gradients
* Weak long-term memory
* Information decay over time

---

* Leads to next major topic:

  * Advanced RNN architectures:

    * LSTM
    * GRU

---

* Core concepts introduced:

  * Seq2Seq forecasting
  * Dense supervision across timesteps
  * Stronger gradient flow
  * Causal models
  * Forecast horizons
  * Windows of windows
  * Sequence outputs
  * Multi-horizon prediction
  * TimeDistributed behavior
  * Long-range forecasting difficulty

---

* Biggest conceptual takeaway:

  * Seq2Seq models improve RNN training by supervising predictions at every timestep instead of only the end, creating richer learning signals and better multi-step forecasting behavior


In [ ]:
my_series = tf.data.Dataset.range(7)
dataset = to_windows(to_windows(my_series, 3), 4)
list(dataset)

In [ ]:
dataset = dataset.map(lambda S: (S[:, 0], S[:, 1:]))
list(dataset)

In [ ]:
def to_seq2seq_dataset(series, seq_length=56, ahead=14, target_col=1,
                       batch_size=32, shuffle=False, seed=None):
    ds = to_windows(tf.data.Dataset.from_tensor_slices(series), ahead + 1)
    ds = to_windows(ds, seq_length).map(
        lambda S: (S[:, 0], S[:, 1:, target_col]))
    if shuffle:
        ds = ds.shuffle(8 * batch_size, seed=seed)
    return ds.batch(batch_size)

In [ ]:
seq2seq_train = to_seq2seq_dataset(mulvar_train, shuffle=True, seed=42)
seq2seq_valid = to_seq2seq_dataset(mulvar_valid)

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
seq2seq_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
    # equivalent: tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(14))
    # also equivalent: tf.keras.layers.Conv1D(14, kernel_size=1)
])

In [ ]:
fit_and_evaluate(seq2seq_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1)

In [ ]:
X = mulvar_valid.to_numpy()[np.newaxis, :seq_length]
y_pred_14 = seq2seq_model.predict(X)[0, -1]  # only the last time step's output

In [ ]:
Y_pred_valid = seq2seq_model.predict(seq2seq_valid)
for ahead in range(14):
    preds = pd.Series(Y_pred_valid[:-1, -1, ahead],
                      index=mulvar_valid.index[56 + ahead : -14 + ahead])
    mae = (preds - mulvar_valid["rail"]).abs().mean() * 1e6
    print(f"MAE for +{ahead + 1}: {mae:,.0f}")

# Handling Long Sequences

To train an RNN on long sequences, we must run it over many time steps,
making the unrolled RNN a very deep network. Just like any deep neural
network it may suffer from the unstable gradients problem, discussed in
Chapter 11: it may take forever to train, or training may be unstable.
Moreover, when an RNN processes a long sequence, it will gradually forget
the first inputs in the sequence. Let’s look at both these problems, starting
with the unstable gradients problem.


## Fighting the Unstable Gradients Problem

* Major RNN problem:

  * Unstable gradients
  * Includes:

    * exploding gradients
    * vanishing gradients

---

* Standard deep learning tricks still help:

  * Good initialization
  * Better optimizers
  * Dropout
  * Smaller learning rates

---

* Important difference from feedforward nets:

  * ReLU is often NOT ideal for RNNs

---

* Why ReLU can destabilize RNNs:

  * Same weights reused across all timesteps
  * Small increase at one timestep propagates repeatedly

---

* Result:

  * Activations may grow exponentially through time

---

* Exploding activations intuition:

$$[
h_1 \uparrow \rightarrow h_2 \uparrow \rightarrow h_3 \uparrow \rightarrow \dots
]$$

* Eventually:

$$[
\text{outputs explode}
]$$

---

* Why `tanh` is default in RNNs:

  * Saturating activation
  * Naturally limits output range

---

* `tanh` output range:

$-1 \leq \tanh(x) \leq 1$

* Helps stabilize recurrent dynamics

---

* Another instability:

  * Exploding gradients

---

* Solution:

  * Gradient clipping

---

* Gradient clipping:

  * Limit gradient magnitude during backpropagation

---

* Purpose:

  * Prevent massive parameter updates
  * Stabilize training

---

* Important monitoring practice:

  * Track gradient sizes using tools like:

    * TensorBoard

---

* Batch Normalization (BN) limitations in RNNs:

  * Works well in feedforward nets
  * Less effective inside recurrent dynamics

---

* Why BN struggles in RNNs:

  * Same BN layer reused at every timestep
  * But sequence statistics may vary over time

---

* Consequence:

  * Poor normalization quality across timesteps

---

* Research finding:

  * BN slightly helps:

    * between recurrent layers

* BN usually hurts:

  * inside recurrent recurrence

---

* Vertical vs horizontal normalization:

  * Vertical:

    * between stacked RNN layers
    * somewhat useful

  * Horizontal:

    * across timesteps inside recurrence
    * usually problematic

---

* Better alternative:

  * Layer Normalization (LN)

---

* Layer Normalization:

  * Introduced by Jimmy Lei Ba

---

* Key difference from BN:

BatchNorm:

* normalizes across:

$$[
\text{batch dimension}
]$$

LayerNorm:

* normalizes across:

$$[
\text{feature dimension}
]$$

---

* Major advantages of LayerNorm in RNNs:

  1. Works independently for each sequence
  2. Works at every timestep
  3. Same behavior during training/testing
  4. No moving averages needed

---

* Important placement in RNN:

  * Applied after linear transformation
  * Before activation

---

* Computation order:

$$[
\text{Linear Combination}
\rightarrow
\text{LayerNorm}
\rightarrow
\text{Activation}
]$$

---

* Custom RNN cell concept:

  * In Keras:

    * RNN cell behaves like custom layer

---

* Cell receives:

  1. Current inputs
  2. Previous hidden states

---

* Important RNN cell attributes:

  * `state_size`
  * `output_size`

---

* SimpleRNN property:

  * Output = hidden state

---

* Layer-normalized RNN workflow:

  1. Compute linear combination
  2. Apply LayerNorm
  3. Apply activation
  4. Return normalized hidden state

---

* Important implementation insight:

  * Activation removed initially from SimpleRNNCell

Reason:

* Need normalization BEFORE activation

---

* Keras flexibility:

  * Custom recurrent cells can easily be created

---

* Simpler regularization option:

  * Built-in dropout support

---

* Two dropout types in RNNs:

1. `dropout`

* Applied to inputs

2. `recurrent_dropout`

* Applied to hidden states between timesteps

---

* Recurrent dropout:

  * Especially useful for regularizing temporal memory

---

* Important forecasting uncertainty technique:

  * MC Dropout (Monte Carlo Dropout)

---

* MC Dropout idea:

  * Keep dropout active during inference

---

* Procedure:

  1. Predict multiple times
  2. Each prediction slightly different
  3. Compute:

  * mean prediction
  * prediction uncertainty

---

* Benefits:

  * Gives:

    * forecast confidence intervals
    * uncertainty estimates
    * error bars

---

* Important inference trick:

```python id="zxx4eq"
model(X, training=True)
```

* Keeps dropout active at inference time

---

* Core concepts introduced:

  * Exploding activations
  * Exploding gradients
  * Saturating activations
  * Gradient clipping
  * BatchNorm limitations in RNNs
  * Layer Normalization
  * Feature-wise normalization
  * Custom recurrent cells
  * Recurrent dropout
  * MC Dropout uncertainty estimation

---

* Biggest conceptual takeaway:

  * RNN stability is much harder than regular deep networks because the same parameters repeatedly interact through time, causing errors and activations to amplify recursively unless carefully controlled with normalization, dropout, and stable activations


In [ ]:
class LNSimpleRNNCell(tf.keras.layers.Layer):
    def __init__(self, units, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = tf.keras.layers.SimpleRNNCell(units,
                                                             activation=None)
        self.layer_norm = tf.keras.layers.LayerNormalization()
        self.activation = tf.keras.activations.get(activation)

    def call(self, inputs, states):
        outputs, new_states = self.simple_rnn_cell(inputs, states)
        norm_outputs = self.activation(self.layer_norm(outputs))
        return norm_outputs, [norm_outputs]

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
custom_ln_model = tf.keras.Sequential([
    tf.keras.layers.RNN(LNSimpleRNNCell(32), return_sequences=True,
                        input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

In [ ]:
fit_and_evaluate(custom_ln_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

### Extra Material – Creating a Custom RNN Class

In [ ]:
class MyRNN(tf.keras.layers.Layer):
    def __init__(self, cell, return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.cell = cell
        self.return_sequences = return_sequences

    def get_initial_state(self, inputs):
        try:
            return self.cell.get_initial_state(inputs)
        except AttributeError:
            # fallback to zeros if self.cell has no get_initial_state() method
            batch_size = tf.shape(inputs)[0]
            return [tf.zeros([batch_size, self.cell.state_size],
                             dtype=inputs.dtype)]

    @tf.function
    def call(self, inputs):
        states = self.get_initial_state(inputs)
        shape = tf.shape(inputs)
        batch_size = shape[0]
        n_steps = shape[1]
        sequences = tf.TensorArray(
            inputs.dtype, size=(n_steps if self.return_sequences else 0))
        outputs = tf.zeros(shape=[batch_size, self.cell.output_size],
                           dtype=inputs.dtype)
        for step in tf.range(n_steps):
            outputs, states = self.cell(inputs[:, step], states)
            if self.return_sequences:
                sequences = sequences.write(step, outputs)

        if self.return_sequences:
            # stack the outputs into an array of shape
            # [time steps, batch size, dims], then transpose it to shape
            # [batch size, time steps, dims]
            return tf.transpose(sequences.stack(), [1, 0, 2])
        else:
            return outputs

Note that `@tf.function` requires the `outputs` variable to be created before the `for` loop, which is why we initialize its value to a zero tensor, even though we don't use that value at all. Once the function is converted to a graph, this unused value will be pruned from the graph, so it doesn't impact performance. Similarly, `@tf.function` requires the `sequences` variable to be created before the `if` statement where it is used, even if `self.return_sequences` is `False`, so we create a `TensorArray` of size 0 in this case.

In [ ]:
tf.random.set_seed(42)

custom_model = tf.keras.Sequential([
    MyRNN(LNSimpleRNNCell(32), return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

In [ ]:
fit_and_evaluate(custom_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

## Tackling the Short-Term Memory Problem

* Major limitation of basic RNNs:

  * Poor long-term memory

---

* Core problem:

  * Information gets transformed repeatedly across timesteps
  * Small information loss happens at every step

---

* Over many timesteps:

  * Early information gradually disappears

---

* Result:

  * Hidden state eventually loses trace of old inputs

---

* Memory decay intuition:

$$[
x_1 \rightarrow h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow \dots
]$$

* Information from:

$$[
x_1
]$$

becomes weaker and weaker over time

---

* Consequence:

  * Basic RNNs struggle with:

    * long sequences
    * long dependencies
    * distant context

---

* Classic example:

  * Language translation

* Problem:

  * Meaning of later words may depend on very early words

---

* Dory analogy:

  * Like Dory forgetting beginning of sentence while reading end of it

---

* This issue becomes severe in:

  * NLP
  * long text
  * speech
  * long time series
  * audio processing

---

* Root technical cause:

  * Repeated transformations + vanishing gradients

---

* Vanishing gradients:

  * During BPTT:

    * gradients shrink exponentially through time

---

* Result:

  * Early timesteps learn extremely slowly
  * Long-term dependencies become hard to capture

---

* Solution:

  * Specialized long-memory cells

---

* Goal of long-memory architectures:

  * Preserve important information across many timesteps

---

* Key idea:

  * Learn:

    * what to remember
    * what to forget
    * what to output

---

* Most successful architecture:

  * LSTM

---

* LSTM:

  * Long Short-Term Memory

---

* Why LSTMs became dominant:

  * Handle long-range dependencies far better than SimpleRNNs
  * More stable gradient flow
  * Better memory preservation

---

* Important historical shift:

  * Basic SimpleRNNs are now rarely used in serious sequence tasks
  * LSTMs and GRUs largely replaced them

---

* Core concepts introduced:

  * Long-term dependency problem
  * Memory decay
  * Vanishing gradients
  * Information loss across time
  * Long-memory architectures
  * LSTM motivation

---

* Biggest conceptual takeaway:

  * Standard RNNs continuously compress history into a small hidden state, causing old information to fade away, so advanced architectures like LSTMs were designed specifically to preserve important information across long sequences


### LSTM cells

* **LSTM (Long Short-Term Memory)**:

  * Introduced in 1997 by:

    * Sepp Hochreiter
    * Jürgen Schmidhuber

---

* Purpose of LSTM:

  * Solve long-term memory problem in basic RNNs
  * Preserve important information across long sequences

---

* Main advantages over SimpleRNN:

  1. Better long-term memory
  2. Faster convergence
  3. More stable training
  4. Better gradient flow
  5. Handles long dependencies effectively

---

* Keras usage:

  * Replace:

```python id="fxm54u"
SimpleRNN
```

with:

```python id="ppjlwm"
LSTM
```

---

* LSTM behaves externally like a normal RNN:

  * sequence input
  * recurrent processing
  * sequence/vector output

* But internally:

  * much more sophisticated memory system

---

* LSTM has TWO states instead of one:

1. Short-term state:

$h^{(t)}$

2. Long-term state:

$c^{(t)}$

---

* Interpretation:

  * ( $h^{(t)}$ ):

    * temporary working memory
    * immediate output

  * ($c^{(t)} $):

    * persistent long-term memory

---

* Core LSTM idea:

  * Learn:

    * what to remember
    * what to forget
    * what to output

---

* LSTM introduces gates:

  1. Forget gate
  2. Input gate
  3. Output gate

---

* Gates:

  * Use sigmoid activation

---

* Sigmoid range:

$$0 \leq \sigma(x) \leq 1$$

* Meaning:

  * 0 → block information
  * 1 → fully allow information

---

* Gates act like:

  * information filters/controllers

---

* Forget gate:

  * Controls what old memory to erase

$$f^{(t)}$$

---

* Input gate:

  * Controls what new information to store

$i^{(t)}$

---

* Output gate:

  * Controls what information becomes output

$o^{(t)}$

---

* Candidate memory:

  * Proposed new memory information

$g^{(t)}$

---

* Memory update process:

1. Forget unnecessary old memory
2. Add important new memory
3. Produce controlled output

---

* Long-term memory update:

$$c^{(t)} = f^{(t)} \otimes c^{(t-1)} + i^{(t)} \otimes g^{(t)}$$

---

* Meaning:

  * Preserve part of old memory
  * Add selected new memory

---

* Final output computation:

$$h^{(t)} = o^{(t)} \otimes \tanh(c^{(t)})$$

---

* Extremely important insight:

  * Long-term memory flows almost directly through time

* This creates:

  * much better gradient propagation
  * less memory decay

---

* Why LSTMs work so well:

  * Information can survive across many timesteps without repeated destructive transformations

---

* What LSTM learns automatically:

  * Important events
  * When to store them
  * How long to keep them
  * When to use them later

---

* Example:

  * In language:

    * remember subject at sentence beginning
    * use it much later

* In time series:

  * remember seasonal or long-range patterns

---

* Important implementation detail:

  * Forget gate bias initialized to:

[
1
]

instead of:

[
0
]

* Purpose:

  * Prevent immediate forgetting during early training

---

* Important architectural insight:

  * SimpleRNN:

$$[
\text{single memory stream}
]$$

* LSTM:

$$[
\text{controlled memory system with gates}
]$$

---

* Main conceptual leap from SimpleRNN:

  * Memory becomes explicitly managed instead of implicitly compressed

---

* Core concepts introduced:

  * Long-term state
  * Short-term state
  * Forget gate
  * Input gate
  * Output gate
  * Candidate memory
  * Controlled memory flow
  * Gated recurrent architecture
  * Gradient preservation

---

* Biggest conceptual takeaway:

  * LSTMs succeed because they transform memory from a fragile continuously overwritten hidden state into a controlled gated storage system that can selectively preserve important information across very long sequences


$$i^{(t)} = \sigma\left(W_{xi}^{\top}x^{(t)} + W_{hi}^{\top}h^{(t-1)} + b_i\right)$$

$$f^{(t)} = \sigma\left(W_{xf}^{\top}x^{(t)} + W_{hf}^{\top}h^{(t-1)} + b_f\right)$$

$$o^{(t)} = \sigma\left(W_{xo}^{\top}x^{(t)} + W_{ho}^{\top}h^{(t-1)} + b_o\right)$$

$$g^{(t)} = \tanh\left(W_{xg}^{\top}x^{(t)} + W_{hg}^{\top}h^{(t-1)} + b_g\right)$$

$$c^{(t)} = f^{(t)} \otimes c^{(t-1)} + i^{(t)} \otimes g^{(t)}$$

$$h^{(t)} = y^{(t)} = o^{(t)} \otimes \tanh\left(c^{(t)}\right)$$


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

In [ ]:
fit_and_evaluate(lstm_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

### GRU cells

* **GRU (Gated Recurrent Unit)**:

  * Proposed in 2014 by Kyunghyun Cho
  * Simplified version of LSTM

---

* Key idea:

  * Achieve LSTM-like performance with:

    * fewer components
    * simpler architecture
    * fewer parameters

---

* Important practical fact:

  * GRUs often perform nearly as well as LSTMs
  * Faster and lighter in many cases

---

* Main simplifications compared to LSTM:

1. Single state vector

* LSTM:

  * short-term state:

$[
h^{(t)}
]$

* long-term state:
$
[
c^{(t)}
]$

* GRU:

  * merges both into:

$h^{(t)}$

---

2. Combined forget + input behavior

* GRU uses:

  * update gate

$z^{(t)}$

---

* Update gate controls:

  * memory preservation
  * memory replacement

---

* Intuition:

  * If:

$[
z^{(t)} \approx 1
]$

→ keep old memory

* If:

$[
z^{(t)} \approx 0
]$

→ replace with new memory

---

3. No output gate

* Entire hidden state becomes output directly

---

4. Reset gate added

$r^{(t)}$

* Controls:

  * how much past information influences candidate memory

---

* Reset gate intuition:

  * Decide how much old memory to ignore when creating new candidate state

---

* GRU equations:

Update gate:

$$z^{(t)} = \sigma\left(W_{xz}^{\top}x^{(t)} + W_{hz}^{\top}h^{(t-1)} + b_z\right)$$

---

Reset gate:

$$r^{(t)} = \sigma\left(W_{xr}^{\top}x^{(t)} + W_{hr}^{\top}h^{(t-1)} + b_r\right)$$

---

Candidate hidden state:

$$g^{(t)} = \tanh\left(W_{xg}^{\top}x^{(t)} + W_{hg}^{\top}(r^{(t)} \otimes h^{(t-1)}) + b_g\right)$$

---

Final hidden state update:

$$h^{(t)} = z^{(t)} \otimes h^{(t-1)} + (1-z^{(t)}) \otimes g^{(t)}$$

---

* Important intuition of final equation:

  * Hidden state becomes mixture of:

    * old memory
    * new candidate memory

---

* If update gate close to 1:

  * Preserve previous state

---

* If update gate close to 0:

  * Replace with new information

---

* Keras implementation:

  * Simply replace:

```python id="8xjdx4"
SimpleRNN
```

or

```python id="nt7m10"
LSTM
```

with:

```python id="b2n75y"
GRU
```

---

* Important comparison:

| Architecture | Complexity | Memory Power | Speed  |
| ------------ | ---------- | ------------ | ------ |
| SimpleRNN    | Low        | Weak         | Fast   |
| LSTM         | High       | Strong       | Slower |
| GRU          | Medium     | Strong       | Faster |

---

* Important limitation:

  * Even LSTM/GRU struggle with:

$$[
100+ \text{ timesteps}
]$$

in very long sequences

---

* Hard tasks:

  * Long audio
  * Long text
  * Long time series

---

* Next idea introduced:

  * Use:

    * 1D Convolutions
  * To shorten/compress sequences before recurrent processing

---

* Core concepts introduced:

  * GRU architecture
  * Update gate
  * Reset gate
  * Unified memory state
  * Simplified gating
  * Memory interpolation
  * Efficient recurrent modeling

---

* Biggest conceptual takeaway:

  * GRUs simplify LSTMs by merging memory systems and reducing gating complexity while still preserving the key ability to selectively retain and update information across long sequences


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
gru_model = tf.keras.Sequential([
    tf.keras.layers.GRU(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

In [ ]:
fit_and_evaluate(gru_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

### Using 1D convolutional layers to process sequences

* **1D Convolution for sequences**:

  * Similar idea to 2D CNNs for images
  * Instead of sliding filters across height & width:

    * slide kernels across time dimension

---

* 1D convolution learns:

  * short local sequential patterns

---

* Kernel/filter:

  * Small sliding window over sequence

* Kernel size determines:

  * maximum local pattern length detected

---

* Example:

  * kernel size:

[
4
]

* Model looks at:

$$[
4 \text{ consecutive timesteps at once}
]$$

---

* Multiple filters:

  * Each filter learns different temporal pattern

---

* Example:

  * 10 filters →

[
10
]

feature maps

* Equivalent to:

$$[
10D \text{ sequence representation}
]$$

---

* Important idea:

  * CNNs can extract temporal features before RNN processing

---

* Hybrid architecture:

  * Conv1D + RNN/GRU/LSTM

---

* Why useful:

  * CNN captures short local patterns efficiently
  * RNN captures longer dependencies

---

* Padding behavior:

1. `"same"` padding

* output length = input length

2. `"valid"` padding

* output becomes shorter

---

* Stride:

  * Controls movement step of kernel

---

* Example:

[
stride = 2
]

* Output sequence length approximately halves

---

* Important consequence:

  * Downsampling occurs

---

* Downsampling:

  * Reduce sequence length
  * Compress temporal information

---

* Why helpful for RNNs:

  * Shorter sequence easier for GRU/LSTM to process
  * Helps detect longer-range dependencies

---

* Architecture used:

  1. Conv1D layer
  2. GRU layer
  3. Dense output layer

---

* Conv1D layer details:

  * Filters:

[
32
]

* Kernel size:

[
4
]

* Stride:

[
2
]

---

* Important insight:

  * Kernel size > stride

* Meaning:

  * overlapping windows
  * all inputs still contribute

---

* Conv layer purpose:

  * Preserve useful information
  * Remove unimportant details

---

* Sequence length increased:

  * From:

[
56
]

to:

[
112
]

days

---

* Why possible now:

  * Conv layer compresses sequence first
  * GRU receives shorter processed representation

---

* Important target adjustment:

  * Conv layer changes temporal alignment

---

* Why crop first 3 timesteps:

  * Kernel size:

[
4
]

* First convolution output depends on:

$$[
t_0 \rightarrow t_3
]$$

* Therefore earliest prediction starts later

---

* Another target adjustment:

  * Stride:

[
2
]

* Output sequence downsampled by factor 2

* Targets must also be downsampled

---

* Key practical insight:

  * When sequence length changes:

    * targets must align correctly

---

* Why Conv1D improves performance:

  1. Reduces sequence length
  2. Extracts local temporal patterns
  3. Makes recurrent learning easier
  4. Helps long-range pattern detection

---

* Important realization:

  * CNNs alone can model sequences surprisingly well

---

* Why CNNs can replace RNNs sometimes:

  * Parallel computation
  * Faster training
  * Good local pattern extraction
  * Large receptive fields possible with deep stacks

---

* Major conceptual shift:

  * Sequential modeling ≠ only RNNs
  * CNNs are also strong sequence learners

---

* Core concepts introduced:

  * 1D convolution
  * Temporal kernels
  * Feature maps
  * Temporal downsampling
  * Stride
  * Sequence compression
  * Hybrid CNN-RNN architectures
  * Receptive field expansion
  * Temporal alignment

---

* Biggest conceptual takeaway:

  * 1D convolutions act like temporal feature extractors that compress and summarize local sequential patterns, making long-range sequence learning easier and sometimes even replacing recurrent networks entirely


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
conv_rnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=4, strides=2,
                           activation="relu", input_shape=[None, 5]),
    tf.keras.layers.GRU(32, return_sequences=True),
    tf.keras.layers.Dense(14)
])

longer_train = to_seq2seq_dataset(mulvar_train, seq_length=112,
                                       shuffle=True, seed=42)
longer_valid = to_seq2seq_dataset(mulvar_valid, seq_length=112)
downsampled_train = longer_train.map(lambda X, Y: (X, Y[:, 3::2]))
downsampled_valid = longer_valid.map(lambda X, Y: (X, Y[:, 3::2]))

In [ ]:
fit_and_evaluate(conv_rnn_model, downsampled_train, downsampled_valid,
                 learning_rate=0.1, epochs=5)

### WaveNet

* **WaveNet**:

  * Introduced in 2016 by Aaron van den Oord and researchers at DeepMind

---

* Core idea:

  * Stack many dilated 1D convolution layers
  * Capture:

    * short-term patterns
    * long-term dependencies
  * Without using RNNs

---

* Major breakthrough:

  * Handle extremely long sequences efficiently

---

* Key concept:

  * **Dilated convolution**

---

* Dilation rate:

  * Controls spacing between inputs used by convolution

---

* Normal convolution:

  * Consecutive inputs used

Example:
$$[
[x_1,x_2]
]$$

---

* Dilated convolution:

  * Inputs spaced apart

Example with dilation=2:
$$[
[x_1,x_3]
]$$

---

* Effect:

  * Receptive field grows rapidly
  * Without huge computational cost

---

* Receptive field:

  * Portion of past sequence visible to neuron

---

* WaveNet dilation pattern:

$$[
1,\ 2,\ 4,\ 8,\ 16,\dots
]$$

* Exponential growth

---

* Important intuition:

  * Lower layers:

    * learn local short-term patterns

* Higher layers:

  * learn broad long-range dependencies

---

* Receptive field growth example:

Layer 1:
$$[
2 \text{ timesteps}
]$$

Layer 2:
$$[
4 \text{ timesteps}
]$$

Layer 3:
$$[
8 \text{ timesteps}
]$$

Layer 4:
$$[
16 \text{ timesteps}
]$$

---

* Major advantage:

  * Huge temporal coverage with few layers

---

* Original WaveNet architecture:

  * 30 layers total
  * Organized into:

$$[
3 \text{ stacks} \times 10 \text{ layers}
]$$

---

* Dilation cycle repeated:
  $$[
  1,2,4,8,\dots,512
  ]$$

---

* Important insight:

  * Equivalent to enormous convolution kernel

---

* Single 10-layer stack behaves roughly like:

$$[
kernel\ size \approx 1024
]$$

* But:

  * far fewer parameters
  * much faster computation

---

* Key architectural benefit:

  * Efficient long-range temporal modeling

---

* Important padding type:

  * `"causal"` padding

---

* Causal padding:

  * Pads only on left/start side

---

* Why important:

  * Prevents model from seeing future information

---

* Prevents temporal leakage:

  * Model remains causal

---

* Causal condition:

Prediction at time:
[
t
]

can only use:
$$[
\leq t
]$$

information

---

* Important contrast:

`same` padding:

* pads both sides

`causal` padding:

* pads only past side

---

* Kernel size used:

[
2
]

* Small kernels + dilation create large receptive field

---

* Final layer:

```python id="5svwbv"
Conv1D(filters=14, kernel_size=1)
```

---

* Kernel size:

[
1
]

acts like:

* Dense layer across features

---

* Important property:

  * Sequence length preserved throughout network

Reason:

* causal padding

---

* Benefit:

  * No need for:

    * cropping
    * downsampling
    * alignment fixes

---

* Why WaveNet powerful:

  1. Parallelizable
  2. Handles huge sequences
  3. Stable gradients
  4. Large receptive fields
  5. No recurrent computation bottleneck

---

* Major difference from RNNs:

RNN:
$$[
\text{Sequential processing}
]$$

WaveNet:
$$[
\text{Parallel convolutional processing}
]$$

---

* Why this matters:

  * Much faster training
  * Better scalability

---

* Extremely important achievement:

  * State-of-the-art audio generation

---

* Audio challenge:

  * 1 second audio can contain:

[
10,000+
]

timesteps

---

* LSTM/GRU struggle:

  * Sequences too long

---

* WaveNet succeeded because:

  * Dilated convolutions efficiently cover huge temporal ranges

---

* Applications:

  * Text-to-speech
  * Audio synthesis
  * Music generation
  * Long time series forecasting

---

* Important real-world warning:

  * Models assume future resembles past

---

* Chicago ridership example:

  * Models failed badly during:

    * COVID-19 pandemic

Reason:

* Learned historical patterns broke completely

---

* Extremely important ML concept:

  * Distribution shift

---

* Distribution shift:

  * Training data distribution changes in real world

---

* Consequences:

  * Forecasting models degrade badly

---

* Practical deployment lesson:

  1. Validate on recent data
  2. Continuously monitor production performance
  3. Retrain when patterns shift

---

* Core concepts introduced:

  * WaveNet
  * Dilated convolution
  * Receptive field
  * Exponential dilation growth
  * Causal padding
  * Temporal leakage prevention
  * Parallel sequence modeling
  * Distribution shift

---

* Biggest conceptual takeaway:

  * WaveNet showed that extremely long sequential dependencies can be modeled efficiently using stacked dilated convolutions, avoiding the memory and sequential-processing limitations of recurrent networks entirely


In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
wavenet_model = tf.keras.Sequential()
wavenet_model.add(tf.keras.layers.InputLayer(input_shape=[None, 5]))
for rate in (1, 2, 4, 8) * 2:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation="relu",
        dilation_rate=rate))
wavenet_model.add(tf.keras.layers.Conv1D(filters=14, kernel_size=1))

In [ ]:
fit_and_evaluate(wavenet_model, longer_train, longer_valid,
                 learning_rate=0.1, epochs=5)

#### wavenet extra

Here is the original WaveNet defined in the paper: it uses Gated Activation Units instead of ReLU and parametrized skip connections, plus it pads with zeros on the left to avoid getting shorter and shorter sequences:

In [ ]:
class GatedActivationUnit(tf.keras.layers.Layer):
    def __init__(self, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.activation = tf.keras.activations.get(activation)

    def call(self, inputs):
        n_filters = inputs.shape[-1] // 2
        linear_output = self.activation(inputs[..., :n_filters])
        gate = tf.keras.activations.sigmoid(inputs[..., n_filters:])
        return self.activation(linear_output) * gate

In [ ]:
def wavenet_residual_block(inputs, n_filters, dilation_rate):
    z = tf.keras.layers.Conv1D(2 * n_filters, kernel_size=2, padding="causal",
                            dilation_rate=dilation_rate)(inputs)
    z = GatedActivationUnit()(z)
    z = tf.keras.layers.Conv1D(n_filters, kernel_size=1)(z)
    return tf.keras.layers.Add()([z, inputs]), z

In [ ]:
tf.random.set_seed(42)

n_layers_per_block = 3  # 10 in the paper
n_blocks = 1  # 3 in the paper
n_filters = 32  # 128 in the paper
n_outputs = 14  # 256 in the paper

inputs = tf.keras.layers.Input(shape=[None, 5])
z = tf.keras.layers.Conv1D(n_filters, kernel_size=2, padding="causal")(inputs)
skip_to_last = []
for dilation_rate in [2**i for i in range(n_layers_per_block)] * n_blocks:
    z, skip = wavenet_residual_block(z, n_filters, dilation_rate)
    skip_to_last.append(skip)

z = tf.keras.activations.relu(tf.keras.layers.Add()(skip_to_last))
z = tf.keras.layers.Conv1D(n_filters, kernel_size=1, activation="relu")(z)
Y_preds = tf.keras.layers.Conv1D(n_outputs, kernel_size=1)(z)

full_wavenet_model = tf.keras.Model(inputs=[inputs], outputs=[Y_preds])

In [ ]:
fit_and_evaluate(full_wavenet_model, longer_train, longer_valid,
                 learning_rate=0.1, epochs=5)

In this chapter we explored the fundamentals of RNNs and used them to process sequences (namely, time series). In the process we also looked at other ways to process sequences, including CNNs. In the next chapter we will use RNNs for Natural Language Processing, and we will learn more about RNNs (bidirectional RNNs, stateful vs stateless RNNs, Encoder–Decoders, and Attention-augmented Encoder-Decoders). We will also look at the Transformer, an Attention-only architecture.

# Exercises

## 1. Can you think of a few applications for a sequence-to-sequence RNN? What about a sequence-to-vector RNN, and a vector-to-sequence RNN?

- Here are a few RNN applications:
    * For a sequence-to-sequence RNN: predicting the weather (or any other time series), machine translation (using an Encoder–Decoder architecture), video captioning, speech to text, music generation (or other sequence generation), identifying the chords of a song
    * For a sequence-to-vector RNN: classifying music samples by music genre, analyzing the sentiment of a book review, predicting what word an aphasic patient is thinking of based on readings from brain implants, predicting the probability that a user will want to watch a movie based on their watch history (this is one of many possible implementations of _collaborative filtering_ for a recommender system)
    * For a vector-to-sequence RNN: image captioning, creating a music playlist based on an embedding of the current artist, generating a melody based on a set of parameters, locating pedestrians in a picture (e.g., a video frame from a self-driving car's camera)

## 2. How many dimensions must the inputs of an RNN layer have? What does each dimension represent? What about its outputs?

An RNN layer must have three-dimensional inputs: the first dimension is the batch dimension (its size is the batch size), the second dimension represents the time (its size is the number of time steps), and the third dimension holds the inputs at each time step (its size is the number of input features per time step). For example, if you want to process a batch containing 5 time series of 10 time steps each, with 2 values per time step (e.g., the temperature and the wind speed), the shape will be [5, 10, 2]. The outputs are also three-dimensional, with the same first two dimensions, but the last dimension is equal to the number of neurons. For example, if an RNN layer with 32 neurons processes the batch we just discussed, the output will have a shape of [5, 10, 32].

## 3. If you want to build a deep sequence-to-sequence RNN, which RNN layers should have return_sequences=True? What about a sequence-to vector RNN?

To build a deep sequence-to-sequence RNN using Keras, you must set `return_sequences=True` for all RNN layers. To build a sequence-to-vector RNN, you must set `return_sequences=True` for all RNN layers except for the top RNN layer, which must have `return_sequences=False` (or do not set this argument at all, since `False` is the default).

## 4. Suppose you have a daily univariate time series, and you want to forecast the next seven days. Which RNN architecture should you use?

If you have a daily univariate time series, and you want to forecast the next seven days, the simplest RNN architecture you can use is a stack of RNN layers (all with `return_sequences=True` except for the top RNN layer), using seven neurons in the output RNN layer. You can then train this model using random windows from the time series (e.g., sequences of 30 consecutive days as the inputs, and a vector containing the values of the next 7 days as the target). This is a sequence-to-vector RNN. Alternatively, you could set `return_sequences=True` for all RNN layers to create a sequence-to-sequence RNN. You can train this model using random windows from the time series, with sequences of the same length as the inputs as the targets. Each target sequence should have seven values per time step (e.g., for time step _t_, the target should be a vector containing the values at time steps _t_ + 1 to _t_ + 7).

## 5. What are the main difficulties when training RNNs? How can you handle them?

The two main difficulties when training RNNs are unstable gradients (exploding or vanishing) and a very limited short-term memory. These problems both get worse when dealing with long sequences. To alleviate the unstable gradients problem, you can use a smaller learning rate, use a saturating activation function such as the hyperbolic tangent (which is the default), and possibly use gradient clipping, Layer Normalization, or dropout at each time step. To tackle the limited short-term memory problem, you can use `LSTM` or `GRU` layers (this also helps with the unstable gradients problem).

## 6. Can you sketch the LSTM cell’s architecture?

An LSTM cell's architecture looks complicated, but it's actually not too hard if you understand the underlying logic. The cell has a short-term state vector and a long-term state vector. At each time step, the inputs and the previous short-term state are fed to a simple RNN cell and three gates: the forget gate decides what to remove from the long-term state, the input gate decides which part of the output of the simple RNN cell should be added to the long-term state, and the output gate decides which part of the long-term state should be output at this time step (after going through the tanh activation function). The new short-term state is equal to the output of the cell.

## 7. Why would you want to use 1D convolutional layers in an RNN?

An RNN layer is fundamentally sequential: in order to compute the outputs at time step _t_, it has to first compute the outputs at all earlier time steps. This makes it impossible to parallelize. On the other hand, a 1D convolutional layer lends itself well to parallelization since it does not hold a state between time steps. In other words, it has no memory: the output at any time step can be computed based only on a small window of values from the inputs without having to know all the past values. Moreover, since a 1D convolutional layer is not recurrent, it suffers less from unstable gradients. One or more 1D convolutional layers can be useful in an RNN to efficiently preprocess the inputs, for example to reduce their temporal resolution (downsampling) and thereby help the RNN layers detect long-term patterns. In fact, it is possible to use only convolutional layers, for example by building a WaveNet architecture.

## 8. Which neural network architecture could you use to classify videos?

To classify videos based on their visual content, one possible architecture could be to take (say) one frame per second, then run every frame through the same convolutional neural network (e.g., a pretrained Xception model, possibly frozen if your dataset is not large), feed the sequence of outputs from the CNN to a sequence-to-vector RNN, and finally run its output through a softmax layer, giving you all the class probabilities. For training you would use cross entropy as the cost function. If you wanted to use the audio for classification as well, you could use a stack of strided 1D convolutional layers to reduce the temporal resolution from thousands of audio frames per second to just one per second (to match the number of images per second), and concatenate the output sequence to the inputs of the sequence-to-vector RNN (along the last dimension).

## 9. Train a classification model for the SketchRNN dataset, available in  TensorFlow Datasets

In [ ]:
tf_download_root = "http://download.tensorflow.org/data/"
filename = "quickdraw_tutorial_dataset_v1.tar.gz"
filepath = tf.keras.utils.get_file(filename,
                                   tf_download_root + filename,
                                   cache_dir=".",
                                   extract=True)

In [ ]:
if "_extracted" in filepath:
    quickdraw_dir = Path(filepath)
else:
    quickdraw_dir = Path(filepath).parent
train_files = sorted(
    [str(path) for path in quickdraw_dir.glob("training.tfrecord-*")]
)
eval_files = sorted(
    [str(path) for path in quickdraw_dir.glob("eval.tfrecord-*")]
)

In [ ]:
train_files

In [ ]:
eval_files

In [ ]:
with open(quickdraw_dir / "eval.tfrecord.classes") as test_classes_file:
    test_classes = test_classes_file.readlines()

with open(quickdraw_dir / "training.tfrecord.classes") as train_classes_file:
    train_classes = train_classes_file.readlines()

In [ ]:
assert train_classes == test_classes
class_names = [name.strip().lower() for name in train_classes]

In [ ]:
sorted(class_names)

In [ ]:
def parse(data_batch):
    feature_descriptions = {
        "ink": tf.io.VarLenFeature(dtype=tf.float32),
        "shape": tf.io.FixedLenFeature([2], dtype=tf.int64),
        "class_index": tf.io.FixedLenFeature([1], dtype=tf.int64)
    }
    examples = tf.io.parse_example(data_batch, feature_descriptions)
    flat_sketches = tf.sparse.to_dense(examples["ink"])
    sketches = tf.reshape(flat_sketches, shape=[tf.size(data_batch), -1, 3])
    lengths = examples["shape"][:, 0]
    labels = examples["class_index"][:, 0]
    return sketches, lengths, labels

In [ ]:
def quickdraw_dataset(filepaths, batch_size=32, shuffle_buffer_size=None,
                      n_parse_threads=5, n_read_threads=5, cache=False):
    dataset = tf.data.TFRecordDataset(filepaths,
                                      num_parallel_reads=n_read_threads)
    if cache:
        dataset = dataset.cache()
    if shuffle_buffer_size:
        dataset = dataset.shuffle(shuffle_buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(parse, num_parallel_calls=n_parse_threads)
    return dataset.prefetch(1)

In [ ]:
train_set = quickdraw_dataset(train_files, shuffle_buffer_size=10000)
valid_set = quickdraw_dataset(eval_files[:5])
test_set = quickdraw_dataset(eval_files[5:])

In [ ]:
for sketches, lengths, labels in train_set.take(1):
    print("sketches =", sketches)
    print("lengths =", lengths)
    print("labels =", labels)

In [ ]:
def draw_sketch(sketch, label=None):
    origin = np.array([[0., 0., 0.]])
    sketch = np.r_[origin, sketch]
    stroke_end_indices = np.argwhere(sketch[:, -1]==1.)[:, 0]
    coordinates = sketch[:, :2].cumsum(axis=0)
    strokes = np.split(coordinates, stroke_end_indices + 1)
    title = class_names[label.numpy()] if label is not None else "Try to guess"
    plt.title(title)
    plt.plot(coordinates[:, 0], -coordinates[:, 1], "y:")
    for stroke in strokes:
        plt.plot(stroke[:, 0], -stroke[:, 1], ".-")
    plt.axis("off")

def draw_sketches(sketches, lengths, labels):
    n_sketches = len(sketches)
    n_cols = 4
    n_rows = (n_sketches - 1) // n_cols + 1
    plt.figure(figsize=(n_cols * 3, n_rows * 3.5))
    for index, sketch, length, label in zip(range(n_sketches), sketches, lengths, labels):
        plt.subplot(n_rows, n_cols, index + 1)
        draw_sketch(sketch[:length], label)
    plt.show()

for sketches, lengths, labels in train_set.take(1):
    draw_sketches(sketches, lengths, labels)

In [ ]:
lengths = np.concatenate([lengths for _, lengths, _ in train_set.take(1000)])
plt.hist(lengths, bins=150, density=True)
plt.axis([0, 200, 0, 0.03])
plt.xlabel("length")
plt.ylabel("density")
plt.show()

In [ ]:
def crop_long_sketches(dataset, max_length=100):
    return dataset.map(lambda inks, lengths, labels: (inks[:, :max_length], labels))

cropped_train_set = crop_long_sketches(train_set)
cropped_valid_set = crop_long_sketches(valid_set)
cropped_test_set = crop_long_sketches(test_set)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(32, kernel_size=5, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(64, kernel_size=5, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(128, kernel_size=3, strides=2, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-2, clipnorm=1.)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy", "sparse_top_k_categorical_accuracy"])
history = model.fit(cropped_train_set, epochs=2,
                    validation_data=cropped_valid_set)

In [ ]:
y_test = np.concatenate([labels for _, _, labels in test_set])
y_probas = model.predict(test_set)

In [ ]:
np.mean(tf.keras.metrics.sparse_top_k_categorical_accuracy(y_test, y_probas))

In [ ]:
n_new = 10
Y_probas = model.predict(sketches)
top_k = tf.nn.top_k(Y_probas, k=5)
for index in range(n_new):
    plt.figure(figsize=(3, 3.5))
    draw_sketch(sketches[index])
    plt.show()
    print("Top-5 predictions:".format(index + 1))
    for k in range(5):
        class_name = class_names[top_k.indices[index, k]]
        proba = 100 * top_k.values[index, k]
        print("  {}. {} {:.3f}%".format(k + 1, class_name, proba))
    print("Answer: {}".format(class_names[labels[index].numpy()]))

In [ ]:
model.save("my_sketchrnn.keras")

## 10. Download the Bach chorales dataset and unzip it. It is composed of 382 chorales composed by Johann Sebastian Bach. Each chorale is 100 to 640 time steps long, and each time step contains 4 integers, where each integer corresponds to a note’s index on a piano (except for the value 0, which means that no note is played). Train a model—recurrent, convolutional, or both—that can predict the next time step (four notes), given a sequence of time steps from a chorale. Then use this model to generate Bach-like music, one note at a time: you can do this by giving the model the start of a chorale and asking it to predict the next time step, then appending these time steps to the input sequence and asking the model for the next note, and so on. Also make sure to check out Google’s Coconet model, which was used for a nice Google doodle about Bach

### Bach Chorales

In [ ]:
filepath = tf.keras.utils.get_file(
    "jsb_chorales.tgz",
    "https://github.com/ageron/data/raw/main/jsb_chorales.tgz",
    cache_dir=".",
    extract=True)

In [ ]:
if "_extracted" in filepath:
    jsb_chorales_dir = Path(filepath) / "jsb_chorales"
else:
    jsb_chorales_dir = Path(filepath).with_name("jsb_chorales")

train_files = sorted(jsb_chorales_dir.glob("train/chorale_*.csv"))
valid_files = sorted(jsb_chorales_dir.glob("valid/chorale_*.csv"))
test_files = sorted(jsb_chorales_dir.glob("test/chorale_*.csv"))

In [ ]:
import pandas as pd

def load_chorales(filepaths):
    return [pd.read_csv(filepath).values.tolist() for filepath in filepaths]

train_chorales = load_chorales(train_files)
valid_chorales = load_chorales(valid_files)
test_chorales = load_chorales(test_files)

In [ ]:
train_chorales[0]

Notes range from 36 (C1 = C on octave 1) to 81 (A5 = A on octave 5), plus 0 for silence:

In [ ]:
notes = set()
for chorales in (train_chorales, valid_chorales, test_chorales):
    for chorale in chorales:
        for chord in chorale:
            notes |= set(chord)

n_notes = len(notes)
min_note = min(notes - {0})
max_note = max(notes)

assert min_note == 36
assert max_note == 81

Let's write a few functions to listen to these chorales (you don't need to understand the details here, and in fact there are certainly simpler ways to do this, for example using MIDI players, but I just wanted to have a bit of fun writing a synthesizer):

In [ ]:
from IPython.display import Audio

def notes_to_frequencies(notes):
    # Frequency doubles when you go up one octave; there are 12 semi-tones
    # per octave; Note A on octave 4 is 440 Hz, and it is note number 69.
    return 2 ** ((np.array(notes) - 69) / 12) * 440

def frequencies_to_samples(frequencies, tempo, sample_rate):
    note_duration = 60 / tempo # the tempo is measured in beats per minutes
    # To reduce click sound at every beat, we round the frequencies to try to
    # get the samples close to zero at the end of each note.
    frequencies = (note_duration * frequencies).round() / note_duration
    n_samples = int(note_duration * sample_rate)
    time = np.linspace(0, note_duration, n_samples)
    sine_waves = np.sin(2 * np.pi * frequencies.reshape(-1, 1) * time)
    # Removing all notes with frequencies ≤ 9 Hz (includes note 0 = silence)
    sine_waves *= (frequencies > 9.).reshape(-1, 1)
    return sine_waves.reshape(-1)

def chords_to_samples(chords, tempo, sample_rate):
    freqs = notes_to_frequencies(chords)
    freqs = np.r_[freqs, freqs[-1:]] # make last note a bit longer
    merged = np.mean([frequencies_to_samples(melody, tempo, sample_rate)
                     for melody in freqs.T], axis=0)
    n_fade_out_samples = sample_rate * 60 // tempo # fade out last note
    fade_out = np.linspace(1., 0., n_fade_out_samples)**2
    merged[-n_fade_out_samples:] *= fade_out
    return merged

def play_chords(chords, tempo=160, amplitude=0.1, sample_rate=44100, filepath=None):
    samples = amplitude * chords_to_samples(chords, tempo, sample_rate)
    if filepath:
        from scipy.io import wavfile
        samples = (2**15 * samples).astype(np.int16)
        wavfile.write(filepath, sample_rate, samples)
        return display(Audio(filepath))
    else:
        return display(Audio(samples, rate=sample_rate))

In [ ]:
for index in range(3):
    play_chords(train_chorales[index])

In order to be able to generate new chorales, we want to train a model that can predict the next chord given all the previous chords. If we naively try to predict the next chord in one shot, predicting all 4 notes at once, we run the risk of getting notes that don't go very well together (believe me, I tried). It's much better and simpler to predict one note at a time. So we will need to preprocess every chorale, turning each chord into an arpegio (i.e., a sequence of notes rather than notes played simultaneuously). So each chorale will be a long sequence of notes (rather than chords), and we can just train a model that can predict the next note given all the previous notes. We will use a sequence-to-sequence approach, where we feed a window to the neural net, and it tries to predict that same window shifted one time step into the future.

We will also shift the values so that they range from 0 to 46, where 0 represents silence, and values 1 to 46 represent notes 36 (C1) to 81 (A5).

And we will train the model on windows of 128 notes (i.e., 32 chords).

Since the dataset fits in memory, we could preprocess the chorales in RAM using any Python code we like, but I will demonstrate here how to do all the preprocessing using tf.data (there will be more details about creating windows using tf.data in the next chapter).

In [ ]:
def create_target(batch):
    X = batch[:, :-1]
    Y = batch[:, 1:] # predict next note in each arpegio, at each step
    return X, Y

def preprocess(window):
    window = tf.where(window == 0, window, window - min_note + 1) # shift values
    return tf.reshape(window, [-1]) # convert to arpegio

def bach_dataset(chorales, batch_size=32, shuffle_buffer_size=None,
                 window_size=32, window_shift=16, cache=True):
    def batch_window(window):
        return window.batch(window_size + 1)

    def to_windows(chorale):
        dataset = tf.data.Dataset.from_tensor_slices(chorale)
        dataset = dataset.window(window_size + 1, window_shift, drop_remainder=True)
        return dataset.flat_map(batch_window)

    chorales = tf.ragged.constant(chorales, ragged_rank=1)
    dataset = tf.data.Dataset.from_tensor_slices(chorales)
    dataset = dataset.flat_map(to_windows).map(preprocess)
    if cache:
        dataset = dataset.cache()
    if shuffle_buffer_size:
        dataset = dataset.shuffle(shuffle_buffer_size)
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(create_target)
    return dataset.prefetch(1)

In [ ]:
train_set = bach_dataset(train_chorales, shuffle_buffer_size=1000)
valid_set = bach_dataset(valid_chorales)
test_set = bach_dataset(test_chorales)

Now let's create the model:

* We could feed the note values directly to the model, as floats, but this would probably not give good results. Indeed, the relationships between notes are not that simple: for example, if you replace a C3 with a C4, the melody will still sound fine, even though these notes are 12 semi-tones apart (i.e., one octave). Conversely, if you replace a C3 with a C\#3, it's very likely that the chord will sound horrible, despite these notes being just next to each other. So we will use an `Embedding` layer to convert each note to a small vector representation (see Chapter 16 for more details on embeddings). We will use 5-dimensional embeddings, so the output of this first layer will have a shape of `[batch_size, window_size, 5]`.
* We will then feed this data to a small WaveNet-like neural network, composed of a stack of 4 `Conv1D` layers with doubling dilation rates. We will intersperse these layers with `BatchNormalization` layers for faster better convergence.
* Then one `LSTM` layer to try to capture long-term patterns.
* And finally a `Dense` layer to produce the final note probabilities. It will predict one probability for each chorale in the batch, for each time step, and for each possible note (including silence). So the output shape will be `[batch_size, window_size, 47]`.

In [ ]:
n_embedding_dims = 5

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_notes, output_dim=n_embedding_dims,
                           input_shape=[None]),
    tf.keras.layers.Conv1D(32, kernel_size=2, padding="causal", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(48, kernel_size=2, padding="causal", activation="relu", dilation_rate=2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(64, kernel_size=2, padding="causal", activation="relu", dilation_rate=4),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(96, kernel_size=2, padding="causal", activation="relu", dilation_rate=8),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.Dense(n_notes, activation="softmax")
])

model.summary()

In [ ]:
optimizer = tf.keras.optimizers.Nadam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
model.fit(train_set, epochs=20, validation_data=valid_set)

I have not done much hyperparameter search, so feel free to iterate on this model now and try to optimize it. For example, you could try removing the `LSTM` layer and replacing it with `Conv1D` layers. You could also play with the number of layers, the learning rate, the optimizer, and so on.

In [ ]:
model.save("my_bach_model.keras")
model.evaluate(test_set)

**Note:** There's no real need for a test set in this exercise, since we will perform the final evaluation by just listening to the music produced by the model. So if you want, you can add the test set to the train set, and train the model again, hopefully getting a slightly better model.

Now let's write a function that will generate a new chorale. We will give it a few seed chords, it will convert them to arpegios (the format expected by the model), and use the model to predict the next note, then the next, and so on. In the end, it will group the notes 4 by 4 to create chords again, and return the resulting chorale.

In [ ]:
def generate_chorale(model, seed_chords, length):
    arpegio = preprocess(tf.constant(seed_chords, dtype=tf.int64))
    arpegio = tf.reshape(arpegio, [1, -1])
    for chord in range(length):
        for note in range(4):
            next_note = model.predict(arpegio, verbose=0).argmax(axis=-1)[:1, -1:]
            arpegio = tf.concat([arpegio, next_note], axis=1)
    arpegio = tf.where(arpegio == 0, arpegio, arpegio + min_note - 1)
    return tf.reshape(arpegio, shape=[-1, 4])

This approach has one major flaw: it is often too conservative. Indeed, the model will not take any risk, it will always choose the note with the highest score, and since repeating the previous note generally sounds good enough, it's the least risky option, so the algorithm will tend to make notes last longer and longer. Pretty boring. Plus, if you run the model multiple times, it will always generate the same melody.

So let's spice things up a bit! Instead of always picking the note with the highest score, we will pick the next note randomly, according to the predicted probabilities. For example, if the model predicts a C3 with 75% probability, and a G3 with a 25% probability, then we will pick one of these two notes randomly, with these probabilities. We will also add a `temperature` parameter that will control how "hot" (i.e., daring) we want the system to feel. A high temperature will bring the predicted probabilities closer together, reducing the probability of the likely notes and increasing the probability of the unlikely ones.

In [ ]:
def generate_chorale_v2(model, seed_chords, length, temperature=1):
    arpegio = preprocess(tf.constant(seed_chords, dtype=tf.int64))
    arpegio = tf.reshape(arpegio, [1, -1])
    for chord in range(length):
        for note in range(4):
            next_note_probas = model.predict(arpegio)[0, -1:]
            rescaled_logits = tf.math.log(next_note_probas) / temperature
            next_note = tf.random.categorical(rescaled_logits, num_samples=1)
            arpegio = tf.concat([arpegio, next_note], axis=1)
    arpegio = tf.where(arpegio == 0, arpegio, arpegio + min_note - 1)
    return tf.reshape(arpegio, shape=[-1, 4])

Let's generate 3 chorales using this new function: one cold, one medium, and one hot (feel free to experiment with other seeds, lengths and temperatures). The code saves each chorale to a separate file. You can run these cells over an over again until you generate a masterpiece!

**Please share your most beautiful generated chorale with me on Twitter @aureliengeron, I would really appreciate it! :))**

In [ ]:
seed_chords = test_chorales[2][:8]
play_chords(seed_chords, amplitude=0.2)

In [ ]:
new_chorale = generate_chorale(model, seed_chords, 56)
play_chords(new_chorale)

In [ ]:
new_chorale_v2_cold = generate_chorale_v2(model, seed_chords, 56, temperature=0.8)
play_chords(new_chorale_v2_cold, filepath="bach_cold.wav")

In [ ]:
new_chorale_v2_medium = generate_chorale_v2(model, seed_chords, 56, temperature=1.0)
play_chords(new_chorale_v2_medium, filepath="bach_medium.wav")

In [ ]:
new_chorale_v2_hot = generate_chorale_v2(model, seed_chords, 56, temperature=1.5)
play_chords(new_chorale_v2_hot, filepath="bach_hot.wav")

Lastly, you can try a fun social experiment: send your friends a few of your favorite generated chorales, plus the real chorale, and ask them to guess which one is the real one!

In [ ]:
play_chords(test_chorales[2][:64], filepath="bach_test_4.wav")